# Nonlinear recourse under competition with immutable features

This revision preserves your immutable-feature lists and separates three scoring options. All model parameters are fitted once and frozen; only `theta_1` (targets) and `theta_2` (recommendations) change during policy training. Immutable features remain inputs to both scoring and policy functions, but never change in `br` or `r`.

| Scorer | Best-response computation | What is guaranteed |
|---|---|---|
| `affine` | Closed-form projection in mutable coordinates | Global minimum Euclidean cost, to numerical precision |
| `quadratic` | Nonlinear concave logit; scalar dual solve | Global minimum cost because the score superlevel set is convex |
| `mlp` | Smooth tanh neural network; multistart SLSQP and KKT polishing | Feasibility and a checked strict local minimum; **global optimality is not certified** |

`q_hat`, `rho_hat`, `br`, `r`, `t_0`, `t_hat`, `t_1`, `C`, `V`, and `lambda_` retain the paper's meanings. `br` and `r` are resulting feature vectors. The complete frozen scoring parameter collection $w_0$ is available in `f_w0.state_dict()`; `theta_1` and `theta_2` belong only to the recourse policy.

The quadratic option provides an exact nonlinear benchmark. The neural option also changes the **response optimization and its derivative**; it does not substitute a local linear projection into a nonlinear scorer. Generic nonsmooth models such as decision trees are not supported by this differentiable implementation.


## How to run one experiment

1. Run the **common setup** cells once. They define functions without loading data or launching experiments.
2. Each experiment contains separate affine and quadratic configurations for every dataset. Edit a particular pair through, for example, `experiment_3_configs["German Credit"]["affine"]`. The uploaded five-seed lists and 500 updates are retained; the grids below incorporate the sensitivity results.
3. Set `experiment_X_selected_datasets` and `experiment_X_selected_models`. For example, `["German Credit", "Adult"]` and `["affine", "quadratic"]` run those four combinations in sequence. Use just `["affine"]` or `["quadratic"]` for one model. Unselected configurations do not load data. The MLP implementation remains available, but the experiment templates select only affine and quadratic models.
4. Set `lambda_values` to choose the objective weights. Experiments 1, 3, 4, 5, 6, and 8 expand these into separately saved configurations; experiment 2 uses its existing lambda sweep, and experiment 7 performs no lambda sweep. Run that experiment's execution cell. The default `"run"` mode performs or reloads preflight and then runs the full configured experiment. `"preflight"` performs checks only, `"load"` displays the current configuration's saved results without training, and `"skip"` does nothing.
5. Rerun unchanged settings to resume checkpoints and skip completed trials. Change a scientific parameter to start a separate saved configuration. Set `retry_failed=True` only to retry recorded failed trials after investigating their errors.

**Results:** your `results_dir="./Results"` is retained. Raw trials, policies, histories, diagnostics, and checkpoints remain in separate experiment/dataset/model/configuration folders. Additionally, `Results/tables/` contains an automatically updated CSV and complete configuration JSON for each run, named with the experiment, dataset, model, readable parameter values, and a configuration ID. Changing a budget experiment's fixed `alpha`, `lambda_`, `tau`, or other scientific setting produces different filenames. Grid and nested solver/model settings are also included in the ID, even if their full values do not fit in the readable filename. Rerunning the same settings updates only those settings' exports.

**Figures:** every newly generated plot goes directly into the single `./figures/` folder, across all experiments and datasets. Names follow `experiment-X-dataset-Y-model-M-parameters-Z-ID-plot.png`. The plot title contains **only the dataset and model**, so you can choose paper numbering later. A matching JSON records the full settings. The `figures_dir` configuration option changes the destination without changing a run's scientific identity.

**Existing runs:** use `export_saved_runs("./Results", figures_dir="./figures")` after setup to export tables and regenerate figures from saved results, including results from earlier notebook versions. This uses their recorded configurations and does not retrain models or load datasets. Historical outputs remain intact. The revised notebook's execution outputs are cleared to avoid displaying old plots with outdated titles.

Use persistent output folders if your runtime is temporary (for example, an already mounted Drive path on Colab). Dependencies are NumPy, pandas, SciPy, scikit-learn, PyTorch, matplotlib, and IPython; Folktables is optional for ACSIncome. Give Me Some Credit uses your local `gmsc_path`. Preflight checks numerical behavior, not statistical adequacy or parameter optimality.

## Common setup

In [ ]:
from dataclasses import dataclass
from pathlib import Path

import copy
import math
import time
import warnings

import numpy as np
import pandas as pd
import torch
from torch import nn
import torch.nn.functional as F
from scipy.optimize import minimize, root
from collections import OrderedDict
from sklearn.metrics import roc_auc_score, log_loss
from scipy.special import expit
from sklearn.model_selection import GroupShuffleSplit
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.neural_network import MLPClassifier
from sklearn.datasets import fetch_openml
import matplotlib.pyplot as plt
from IPython.display import display

torch.set_default_dtype(torch.float64)
torch.set_num_threads(1)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 160)


def as_tensor(X):
    return torch.as_tensor(X, dtype=torch.float64, device="cpu")


def generate_population_nd(n_samples, w_star, noise_std=0.0, seed=42):
    rng = np.random.default_rng(seed)
    w_star = np.asarray(w_star, dtype=float)
    X = rng.normal(size=(n_samples, len(w_star)))
    X += rng.normal(scale=noise_std, size=X.shape)
    return X, expit(X @ w_star)


def generate_correlated_nonlinear_population(
    n_samples, correlation=0.8, d=3, noise_std=0.0, seed=42
):
    if d < 2 or not (-1 / (d - 1) < correlation < 1):
        raise ValueError("The correlation matrix must be positive definite; d >= 2.")
    rng = np.random.default_rng(seed)
    covariance = (1 - correlation) * np.eye(d) + correlation * np.ones((d, d))
    X = rng.multivariate_normal(np.zeros(d), covariance, n_samples)
    X += rng.normal(scale=noise_std, size=X.shape)
    w_star = np.array([1.0, -0.5] + [0.0] * (d - 2))
    w_star /= np.linalg.norm(w_star)
    f_star_scores = expit(X @ w_star + 0.8 * np.sin(np.pi * X[:, 0]) * X[:, 1])
    return X, f_star_scores, w_star


def split_indices(X, seed=42):
    """Split before preprocessing/sampling; identical feature rows stay together."""
    frame = pd.DataFrame(X).reset_index(drop=True)
    groups = pd.util.hash_pandas_object(frame, index=False).to_numpy()
    indices = np.arange(len(frame))

    def split(indices, fraction, random_state):
        splitter = GroupShuffleSplit(n_splits=1, test_size=fraction, random_state=random_state)
        left, right = next(splitter.split(indices, groups=groups[indices]))
        return indices[left], indices[right]

    development, test = split(indices, 0.20, seed)
    development, validation = split(development, 0.25, seed + 1)
    fitting, reference = split(development, 0.50, seed + 2)
    result = dict(fit=fitting, train=reference, val=validation, test=test)
    names = list(result)
    for i, left in enumerate(names):
        for right in names[i + 1:]:
            assert set(groups[result[left]]).isdisjoint(groups[result[right]])
    return result


def prepare_population(X, f_star_scores, seed=42, immutable_indices=()):
    """Synthetic population: no sample-dependent rescaling or bootstrap duplicates."""
    X = np.asarray(X, dtype=float)
    f_star_scores = np.asarray(f_star_scores, dtype=float).reshape(-1)
    indices = split_indices(X, seed)
    population = {f"X_{name}": X[idx] for name, idx in indices.items()}
    population["f_star_fit"] = f_star_scores[indices["fit"]]
    population["indices"] = indices
    population["actionable_mask"] = np.ones(X.shape[1], dtype=bool)
    population["actionable_mask"][list(immutable_indices)] = False
    population["feature_names"] = [f"x_{j}" for j in range(X.shape[1])]
    return population


def prepare_real_population(X, y, n_samples=None, seed=42, immutable_features=None, proxy_options=None):
    """Group-disjoint records; preprocessing and qualification proxy fit on fit only.

    n_samples is a cap per partition, sampled WITHOUT replacement.
    Recourse later treats all resulting encoded coordinates as continuous.
    """
    X = pd.DataFrame(X).reset_index(drop=True)
    y = np.asarray(y, dtype=int).reshape(-1)
    indices = split_indices(X, seed)
    if n_samples is not None:
        if n_samples < 2:
            raise ValueError("n_samples must be at least two.")
        rng = np.random.default_rng(seed)
        indices = {name: rng.choice(idx, min(len(idx), n_samples), replace=False)
                   for name, idx in indices.items()}
    numeric = X.select_dtypes(include="number").columns.tolist()
    categorical = [column for column in X if column not in numeric]
    preprocessor = ColumnTransformer([
        ("numeric", Pipeline([
            ("imputer", SimpleImputer(strategy="median", keep_empty_features=True)),
            ("scaler", StandardScaler()),
        ]), numeric),
        ("categorical", Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent", keep_empty_features=True)),
            ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
        ]), categorical),
    ])
    X_fit = preprocessor.fit_transform(X.iloc[indices["fit"]])
    if len(np.unique(y[indices["fit"]])) != 2:
        raise ValueError("The fitting partition must contain both outcome classes.")
    proxy_options = dict(proxy_options or {})
    qualification_proxy = MLPClassifier(
        hidden_layer_sizes=tuple(proxy_options.get("hidden", [32,16])), activation="tanh",
        max_iter=proxy_options.get("max_iter",1500), early_stopping=True,
        random_state=seed,
    ).fit(X_fit, y[indices["fit"]])
    population = {f"X_{name}": np.asarray(preprocessor.transform(X.iloc[idx]), dtype=float)
                  for name, idx in indices.items()}
    population["f_star_fit"] = qualification_proxy.predict_proba(X_fit)[:, 1]

    immutable_features = set(immutable_features or [])
    missing = immutable_features - set(X.columns)
    if missing:
        raise ValueError(f"Immutable columns were not found: {sorted(missing)}")
    # Exact source-column mapping, not string-prefix matching on one-hot names.
    source_features = list(numeric)
    if categorical:
        encoder = preprocessor.named_transformers_["categorical"].named_steps["encoder"]
        source_features += [column for column, categories in zip(categorical, encoder.categories_)
                            for _ in categories]
    feature_names = preprocessor.get_feature_names_out()
    assert len(source_features) == len(feature_names) == X_fit.shape[1]
    population["actionable_mask"] = np.array([name not in immutable_features for name in source_features])
    population["feature_names"] = feature_names.tolist()
    population["encoded_source_features"] = source_features
    population["immutable_features"] = sorted(immutable_features)
    population["y_test"] = y[indices["test"]]
    population.update(indices=indices, preprocessor=preprocessor,
                      qualification_proxy=qualification_proxy)
    return population


def prepare_adult_data(n_samples=None, seed=42, proxy_options=None):
    data = fetch_openml(name="adult", version=2, as_frame=True)
    y = data.target.astype(str).str.rstrip(".").eq(">50K").to_numpy(dtype=int)
    immutable_features = ['age', 'sex', 'race']
    return prepare_real_population(data.data, y, n_samples, seed, immutable_features=immutable_features, proxy_options=proxy_options)


def prepare_credit_data(n_samples=None, seed=42, proxy_options=None):
    data = fetch_openml(data_id=31, as_frame=True)
    y = data.target.astype(str).eq("good").to_numpy(dtype=int)
    immutable_features = ['age', 'personal_status', 'foreign_worker', 'num_dependents']
    return prepare_real_population(data.data, y, n_samples, seed, immutable_features=immutable_features, proxy_options=proxy_options)




def prepare_gmsc_data(path="GiveMeSomeCredit.csv", n_samples=None, seed=42, proxy_options=None):
    frame = pd.read_csv(path)
    y = (frame["SeriousDlqin2yrs"] == 0).to_numpy(dtype=int)
    X = frame.drop(columns=["SeriousDlqin2yrs", "Unnamed: 0"], errors="ignore")
    immutable_features = ['age', 'NumberOfDependents']
    return prepare_real_population(X, y, n_samples, seed, immutable_features=immutable_features, proxy_options=proxy_options)


def prepare_folktables_data(n_samples=None, seed=42, proxy_options=None):
    from folktables import ACSDataSource, ACSIncome  # Optional dependency; install explicitly.
    source = ACSDataSource(survey_year="2018", horizon="1-Year", survey="person")
    frame = source.get_data(states=["CA"], download=True)
    features, labels, _ = ACSIncome.df_to_numpy(frame)
    X = pd.DataFrame(features, columns=ACSIncome.features)
    for column in ["COW", "SCHL", "MAR", "OCCP", "POBP", "RELP", "SEX", "RAC1P"]:
        if column in X:
            X[column] = X[column].astype("category")

    immutable_features = ['AGEP', 'SEX', 'RAC1P', 'MAR', 'RELP', 'POBP']
    return prepare_real_population(X, labels.astype(int), n_samples, seed, immutable_features=immutable_features, proxy_options=proxy_options)

## Data and immutable coordinates

Splits precede preprocessing and sampling; duplicate feature rows remain in the same split. Preprocessing and the real-data qualification proxy use fitting data only. Every encoded column maps to its original feature name exactly, so all one-hot columns of an immutable feature are frozen. A similarly prefixed but different feature is not accidentally frozen.

The user-specified immutable sets are retained: Adult (`age`, `sex`, `race`); German Credit (`age`, `personal_status`, `foreign_worker`, `num_dependents`); Give Me Some Credit (`age`, `NumberOfDependents`); ACSIncome (`AGEP`, `SEX`, `RAC1P`, `MAR`, `RELP`, `POBP`). Synthetic `x_0` is immutable and affects qualification.

Mutable encoded coordinates still use a continuous Euclidean relaxation. Integer, one-hot validity for mutable categories, causal constraints, and feature bounds are not implemented. Thus immutable-feature enforcement should not be confused with complete real-world actionability. Real-data qualification scores come from a learned proxy, not known ground truth.

## Scorers and reachability

The quadratic logit is $w_{0,0}+w_{0,1:d}^{\top}x-\frac12\sum_j h_jx_j^2$, with $h_j>0$. Its nonlinear score has convex superlevel sets when immutable coordinates are fixed. The neural logit is a linear residual plus a two-layer tanh network. The scorer fit retains your L2 regularization; this is regularized empirical fitting rather than a claim of exact population ERM.

For affine/quadratic models, `q_bar` is the globally reachable score. For neural models, budget-constrained multistart search provides a feasible `witness`, which proves attainability of a score but **does not prove a global maximum**; `q_bar` is therefore `None`. `q_cap` is a conservative target cap below the witness score, with a small interior margin to avoid singular endpoint derivatives. The learned targets and all comparison policies use this same restricted target range. Failure to find a high enough neural witness means *unconfirmed eligibility*, not certified infeasibility. The eligible set remains fixed while a policy is trained.

In [ ]:
class FixedScorer(nn.Module):
    """Affine logit; immutable coordinates still contribute to the decision."""
    model_type = "affine"
    def __init__(self, w_0):
        super().__init__()
        self.register_buffer("w_0", as_tensor(w_0).reshape(-1).detach().clone())
        self.register_buffer("h", torch.zeros(len(self.w_0) - 1))

    def logits(self, X):
        return self.w_0[0] + X @ self.w_0[1:]

    def forward(self, X):
        return torch.sigmoid(self.logits(X))


class QuadraticScorer(FixedScorer):
    """Nonlinear score; concave logit gives convex target-attainment sets."""
    model_type = "quadratic"
    def __init__(self, w_0, h):
        super().__init__(w_0)
        self.h.copy_(as_tensor(h))
        if not (self.h > 0).all():
            raise ValueError("Quadratic curvatures must be strictly positive.")

    def logits(self, X):
        return super().logits(X) - 0.5 * (X.square() * self.h).sum(-1)


class SmoothMLPScorer(nn.Module):
    """Smooth nonlinear residual network; no global recourse-optimality guarantee."""
    model_type = "mlp"
    def __init__(self, w_0, hidden=12, seed=42):
        super().__init__()
        with torch.random.fork_rng():
            torch.manual_seed(seed)
            self.w_0 = nn.Parameter(as_tensor(w_0).detach().clone())
            self.network = nn.Sequential(nn.Linear(len(w_0)-1, hidden), nn.Tanh(),
                                         nn.Linear(hidden, hidden), nn.Tanh(), nn.Linear(hidden, 1))
            nn.init.zeros_(self.network[-1].weight)
            nn.init.zeros_(self.network[-1].bias)

    def logits(self, X):
        return self.w_0[0] + X @ self.w_0[1:] + self.network(X).squeeze(-1)

    def forward(self, X):
        return torch.sigmoid(self.logits(X))


def fit_f_w0(X_fit, f_star_scores, model_type="quadratic", seed=42, l2_reg=1e-2, epochs=1000, lr=0.01, hidden=12):
    """Fit once on the fitting split; all returned scoring parameters are frozen."""
    X = np.asarray(X_fit, dtype=float)
    y = np.asarray(f_star_scores, dtype=float).reshape(-1)
    if X.ndim != 2 or len(X) != len(y) or not np.isfinite(X).all() or not np.isfinite(y).all():
        raise ValueError("Invalid fitting data.")
    if not ((0 <= y) & (y <= 1)).all() or l2_reg < 0:
        raise ValueError("Require qualification scores in [0,1] and nonnegative regularization.")
    d = X.shape[1]
    X_aug = np.column_stack([np.ones(len(X)), X])
    design = np.column_stack([X_aug, -0.5 * X**2]) if model_type == "quadratic" else X_aug
    bounds = [(-5, 5)] * (d + 1)
    initial = np.zeros(d + 1)
    if model_type == "quadratic":
        bounds += [(1e-4, 3.0)] * d
        initial = np.r_[initial, np.full(d, 0.05)]

    def objective(parameters):
        logits = design @ parameters
        penalty = parameters.copy(); penalty[0] = 0
        loss = np.mean(np.logaddexp(0.0, logits) - y * logits) + l2_reg * np.dot(penalty, penalty) / 2
        grad = design.T @ (expit(logits) - y) / len(y) + l2_reg * penalty
        return loss, grad

    fitted = minimize(objective, initial, jac=True, method="L-BFGS-B", bounds=bounds,
                      options={"maxiter": 10000, "ftol": 1e-12, "gtol": 1e-8})
    if not fitted.success:
        raise RuntimeError(f"Initial scoring fit failed: {fitted.message}")
    if model_type == "affine":
        f_w0 = FixedScorer(fitted.x)
    elif model_type == "quadratic":
        f_w0 = QuadraticScorer(fitted.x[:d+1], fitted.x[d+1:])
    elif model_type == "mlp":
        f_w0 = SmoothMLPScorer(fitted.x, hidden=hidden, seed=seed)
        optimizer = torch.optim.Adam(f_w0.parameters(), lr=lr)
        X_tensor, y_tensor = as_tensor(X), as_tensor(y)
        for _ in range(epochs):
            optimizer.zero_grad()
            loss = F.binary_cross_entropy_with_logits(f_w0.logits(X_tensor), y_tensor)
            loss = loss + 0.5 * l2_reg * sum(p.square().sum() for p in f_w0.parameters())
            loss.backward(); optimizer.step()
        f_w0.fit_loss = float(loss.detach())
    else:
        raise ValueError("model_type must be affine, quadratic, or mlp.")
    f_w0.eval().requires_grad_(False)
    return f_w0


def cost(X, X_prime):
    return torch.linalg.vector_norm(X_prime - X, dim=-1)


def project_actions(delta, B, actionable_mask):
    delta = delta * actionable_mask
    norms = torch.linalg.vector_norm(delta, dim=-1, keepdim=True)
    return delta * torch.clamp(B / norms.clamp_min(torch.finfo(delta.dtype).tiny), max=1.0)


@torch.no_grad()
def exact_reachable(X, f_w0, B, actionable_mask):
    """Global maximizer within the mutable-coordinate Euclidean ball."""
    a = (f_w0.w_0[1:] - X * f_w0.h) * actionable_mask
    if B == 0 or not actionable_mask.any():
        return X.clone(), f_w0(X)
    if f_w0.model_type == "affine":
        delta = B * a / torch.linalg.vector_norm(a, dim=1, keepdim=True).clamp_min(1e-300)
    else:
        h = f_w0.h
        unconstrained = a / h
        low = torch.zeros(len(X))
        high = torch.linalg.vector_norm(a, dim=1) / B
        for _ in range(75):
            multiplier = (low + high) / 2
            delta = a / (h + multiplier[:, None])
            too_far = torch.linalg.vector_norm(delta, dim=1) > B
            low = torch.where(too_far, multiplier, low)
            high = torch.where(too_far, high, multiplier)
        delta = a / (h + high[:, None])
        delta = torch.where((torch.linalg.vector_norm(unconstrained, dim=1) <= B)[:, None],
                            unconstrained, delta)
    witness = X + delta
    return witness, f_w0(witness)


def numerical_reachable(X, f_w0, B, actionable_mask, seed=42, starts=3, steps=60):
    """A feasible witness/lower bound on the reachable maximum, never a global certificate."""
    if B == 0 or not actionable_mask.any():
        return X.clone(), f_w0(X).detach()
    generator = torch.Generator().manual_seed(seed)
    best_X, best_score = X.clone(), f_w0(X).detach()
    for start in range(starts):
        delta = torch.zeros_like(X) if start == 0 else project_actions(
            torch.randn(X.shape, generator=generator), B, actionable_mask)
        for step in range(steps):
            with torch.enable_grad():
                delta = delta.detach().requires_grad_(True)
                gradient, = torch.autograd.grad(f_w0.logits(X + delta).sum(), delta)
            gradient *= actionable_mask
            gradient /= torch.linalg.vector_norm(gradient, dim=1, keepdim=True).clamp_min(1e-14)
            with torch.no_grad():
                delta = project_actions(delta + (0.25 * B / math.sqrt(1 + step/10)) * gradient,
                                        B, actionable_mask)
                score = f_w0(X + delta)
                better = score > best_score
                best_X[better], best_score[better] = (X + delta)[better], score[better]
    return best_X.detach(), best_score.detach()


@dataclass
class Cohort:
    X: torch.Tensor
    X_aug: torch.Tensor
    f_w0: nn.Module
    t_0: float
    B: float
    alpha: float
    s_0: torch.Tensor
    q_bar: object  # Exact maximum only for affine/quadratic; None for MLP.
    q_cap: torch.Tensor  # Known-feasible target cap used by every comparison method.
    witness: torch.Tensor
    X_minus: torch.Tensor
    eligible: torch.Tensor
    N: int
    N_minus: int
    k: int
    actionable_mask: torch.Tensor
    response_cache: object
    solver_audit: list
    options: dict


def make_cohort(X, f_w0, t_0, B, alpha, actionable_mask=None, seed=42, solver_options=None):
    options = dict(solver_options or {})
    X = as_tensor(X).detach().clone()
    if X.ndim != 2 or not len(X) or not torch.isfinite(X).all():
        raise ValueError("X must be a nonempty finite matrix.")
    if not (0 < t_0 < 1 and 0 < alpha < 1 and np.isfinite(B) and B >= 0):
        raise ValueError("Invalid t_0, alpha, or B.")
    mask = torch.ones(X.shape[1], dtype=torch.bool) if actionable_mask is None else torch.as_tensor(actionable_mask, dtype=torch.bool)
    if mask.shape != (X.shape[1],):
        raise ValueError("actionable_mask must match the encoded feature dimension.")
    s_0 = f_w0(X).detach()
    if not ((0 < s_0) & (s_0 < 1)).all():
        raise FloatingPointError("Scoring saturation: rescale features or strengthen scorer regularization.")
    if getattr(f_w0, "model_type", None) in {"affine", "quadratic"}:
        witness, q_reached = exact_reachable(X, f_w0, B, mask)
        q_bar = q_reached.clone()
        slack = options.get("exact_margin", 1e-4)
    else:
        witness, q_reached = numerical_reachable(X, f_w0, B, mask, seed=seed,
                                                   starts=options.get("reachability_starts",3),
                                                   steps=options.get("reachability_steps",60))
        q_bar, slack = None, options.get("neural_margin",0.02)
    q_cap = torch.minimum(s_0 + (1-slack) * (q_reached-s_0), torch.full_like(s_0, 1-1e-8))
    q_cap = torch.maximum(q_cap, s_0)
    X_minus = s_0 < t_0
    eligible = X_minus & (q_cap >= t_0)
    assert torch.equal(witness[:, ~mask], X[:, ~mask])
    assert (cost(X, witness) <= B + 1e-9).all()
    return Cohort(X, torch.cat([torch.ones((len(X), 1)), X], 1), f_w0, float(t_0), float(B),
                  float(alpha), s_0, q_bar, q_cap, witness, X_minus, eligible, len(X),
                  int(X_minus.sum()), math.floor(alpha*len(X)), mask, OrderedDict(), [], options)


def target_feasible(cohort, q):
    q = as_tensor(q).expand(cohort.N)
    return cohort.X_minus & (q >= cohort.t_0) & (q <= cohort.q_cap) & (q < 1)

## Differentiable minimum-cost responses

Optimization is performed only in mutable coordinates; immutable values are copied exactly. Minimizing half the squared Euclidean norm has the same minimizer as Euclidean cost. For a smooth neural logit $h$, an interior-budget response solves the KKT equations
$\delta-\mu\nabla h(x+\delta)=0$ and $h(x+\delta)=\operatorname{logit}(q)$.
The layer differentiates these equations to obtain $d\operatorname{br}/dq$.

Every accepted numerical response passes target, budget, KKT residual, reduced-Hessian, and conditioning checks. Multiple fixed starts and deterministic branch selection avoid dependence on previous calls. Numerical failure is recorded as failure, not converted into “infeasible,” zero cost/validity, or an invented gradient. For a general neural network, local branches can switch and violate the smoothness assumptions of the paper's theorem. Report empirical diagnostics rather than extending that theorem without additional assumptions.

The optimization-layer approach follows the principle of differentiating through a constrained solver; see [Differentiable Convex Optimization Layers](https://papers.nips.cc/paper_files/paper/2019/hash/9ce3c52fc54362e22053399d3181c638-Abstract.html). That paper's convex guarantees do not make the generic neural response problem convex. The local solver uses [SciPy SLSQP](https://docs.scipy.org/doc/scipy/reference/optimize.minimize-slsqp.html).

In [ ]:
class QuadraticBR(torch.autograd.Function):
    @staticmethod
    def forward(ctx, q, cohort):
        active = target_feasible(cohort, q)
        response, derivative = cohort.X.clone(), torch.zeros_like(cohort.X)
        if active.any():
            X, target = cohort.X[active], q[active]
            h = cohort.f_w0.h
            a = (cohort.f_w0.w_0[1:] - X*h) * cohort.actionable_mask
            gain = torch.logit(target) - cohort.f_w0.logits(X)
            if cohort.f_w0.model_type == "affine":
                squared_norm = a.square().sum(1)
                if (squared_norm <= 0).any():
                    raise ArithmeticError("No mutable score-increasing direction.")
                delta = gain[:, None] * a / squared_norm[:, None]
                jacobian = a / squared_norm[:, None] / (target*(1-target))[:, None]
            else:
                def at(multiplier):
                    delta = multiplier[:, None]*a / (1+multiplier[:, None]*h)
                    achieved = (a*delta - 0.5*h*delta.square()).sum(1)
                    return delta, achieved
                low, high = torch.zeros_like(target), torch.ones_like(target)
                for _ in range(70):
                    insufficient = at(high)[1] < gain
                    if not insufficient.any():
                        break
                    high = torch.where(insufficient, high*2, high)
                else:
                    raise ArithmeticError("Unable to bracket quadratic response multiplier.")
                for _ in range(75):
                    middle = (low+high)/2
                    insufficient = at(middle)[1] < gain
                    low = torch.where(insufficient, middle, low)
                    high = torch.where(insufficient, high, middle)
                delta, _ = at(high)
                slope = a/(1+high[:, None]*h).square()
                denominator = ((a-h*delta)*slope).sum(1)
                if (denominator <= 1e-18).any():
                    raise FloatingPointError("Degenerate best response; increase the target safety margin.")
                jacobian = slope/denominator[:, None]/(target*(1-target))[:, None]
            response[active], derivative[active] = X + delta, jacobian
        ctx.save_for_backward(derivative)
        return response

    @staticmethod
    def backward(ctx, grad_output):
        derivative, = ctx.saved_tensors
        return (grad_output*derivative).sum(1), None


def solve_local_br(cohort, index, q):
    """Multistart constrained solve + KKT polishing; returns a certified local branch.

    No fallback to an unchecked direction, interpolation, or fabricated gradient.
    The witness proves this target is feasible even if numerical optimization fails.
    """
    key = (int(index), float(q))
    if key in cohort.response_cache:
        return cohort.response_cache[key]
    x = cohort.X[index].detach()
    mask = cohort.actionable_mask
    mutable = torch.where(mask)[0]
    d = len(mutable)
    logit_q = math.log(q) - math.log1p(-q)
    witness_delta = (cohort.witness[index]-x)[mask].numpy()
    cache = {}

    def derivatives(delta, hessian=False):
        signature = (np.asarray(delta, dtype=float).tobytes(), hessian)
        if signature in cache:
            return cache[signature]
        with torch.enable_grad():
            u = as_tensor(delta).detach().clone().requires_grad_(True)
            def logit(u):
                z = x.clone(); z[mutable] = x[mutable]+u
                return cohort.f_w0.logits(z[None])[0]
            value = logit(u)
            gradient, = torch.autograd.grad(value, u, create_graph=hessian)
            H = torch.autograd.functional.hessian(logit, u, vectorize=True).detach().numpy() if hessian else None
        answer = (float(value.detach()), gradient.detach().numpy(), H)
        if len(cache) > 100:
            cache.clear()
        cache[signature] = answer
        return answer

    # Deterministic feasible ray start; continuity suffices, no monotonicity is assumed.
    low, high = 0.0, 1.0
    for _ in range(55):
        middle = (low+high)/2
        if derivatives(middle*witness_delta)[0] >= logit_q:
            high = middle
        else:
            low = middle
    ray = high*witness_delta
    constraints = [
        {"type": "ineq", "fun": lambda u: derivatives(u)[0]-logit_q,
         "jac": lambda u: derivatives(u)[1]},
        {"type": "ineq", "fun": lambda u: cohort.B**2-np.dot(u,u),
         "jac": lambda u: -2*u},
    ]
    candidates = []
    # Fixed starts make the branch-selection rule reproducible and independent of training history.
    for initial in [ray, np.zeros(d), witness_delta]:
        result = minimize(lambda u: 0.5*np.dot(u,u), initial, jac=lambda u: u,
                          constraints=constraints, method="SLSQP",
                          options={"ftol": cohort.options.get("response_ftol",1e-11), "maxiter": cohort.options.get("response_maxiter",150)})
        delta = result.x
        value, g, H = derivatives(delta, True)
        if np.dot(g,g) < 1e-16:
            continue
        multiplier = np.dot(delta,g)/np.dot(g,g)

        def equations(z):
            value, gradient, H = derivatives(z[:-1], True)
            return np.r_[z[:-1]-z[-1]*gradient, value-logit_q]

        def jacobian(z):
            _, gradient, H = derivatives(z[:-1], True)
            return np.block([[np.eye(d)-z[-1]*H, -gradient[:,None]],
                             [gradient[None,:], np.zeros((1,1))]])

        polished = root(equations, np.r_[delta,multiplier], jac=jacobian,
                        method="hybr", options={"xtol": 1e-10})
        z = polished.x
        residual = float(np.max(np.abs(equations(z))))
        delta, multiplier = z[:-1], z[-1]
        value, g, H = derivatives(delta, True)
        K = jacobian(z)
        norm = np.linalg.norm(delta)
        if (residual > 2e-7 or multiplier <= 0 or norm > np.linalg.norm(ray)+1e-7 or norm > cohort.B+1e-9
                or norm >= cohort.B-1e-9 or abs(value-logit_q) > 1e-9):
            continue
        tangent = np.linalg.svd(g[None,:], full_matrices=True)[2][1:].T
        reduced = tangent.T @ (np.eye(d)-multiplier*H) @ tangent
        minimum_eigenvalue = np.linalg.eigvalsh(reduced).min() if reduced.size else math.inf
        condition = np.linalg.cond(K)
        if minimum_eigenvalue <= 1e-8 or not np.isfinite(condition) or condition > 1e12:
            continue
        rhs = np.r_[np.zeros(d), 1/(q*(1-q))]
        sensitivity = np.linalg.solve(K, rhs)[:-1]
        candidates.append((norm, delta.copy(), sensitivity, residual, condition))
    if not candidates:
        raise RuntimeError(f"Nonlinear br failed local-optimality/gradient checks at applicant {index}; "
                           "the target has a feasible witness. Increase solver effort or use the quadratic model.")
    _, delta, sensitivity, residual, condition = min(candidates, key=lambda item:item[0])
    response, dq = x.clone(), torch.zeros_like(x)
    response[mask] += as_tensor(delta); dq[mask] = as_tensor(sensitivity)
    audit = dict(target_error=abs(float(cohort.f_w0(response[None])[0])-q),
                 kkt_residual=residual, kkt_condition=condition,
                 local_cost_spread=max(item[0] for item in candidates)-min(item[0] for item in candidates),
                 starts_passed=len(candidates), global_certificate=False)
    answer = (response, dq, audit)
    cohort.solver_audit.append(audit)
    cohort.response_cache[key] = answer
    if len(cohort.response_cache) > 4096:
        cohort.response_cache.popitem(last=False)
    return answer


class NonlinearBR(torch.autograd.Function):
    @staticmethod
    def forward(ctx, q, cohort):
        response, derivative = cohort.X.clone(), torch.zeros_like(cohort.X)
        for index in torch.where(target_feasible(cohort,q))[0].tolist():
            response[index], derivative[index], _ = solve_local_br(cohort,index,float(q[index]))
        ctx.save_for_backward(derivative)
        return response

    @staticmethod
    def backward(ctx, grad_output):
        derivative, = ctx.saved_tensors
        return (grad_output*derivative).sum(1), None


def br(cohort, q):
    q = as_tensor(q).expand(cohort.N)
    layer = QuadraticBR if cohort.f_w0.model_type in {"affine","quadratic"} else NonlinearBR
    response = layer.apply(q,cohort)
    if not torch.equal(response[:,~cohort.actionable_mask],cohort.X[:,~cohort.actionable_mask]):
        raise AssertionError("Immutable features changed.")
    if (cost(cohort.X,response).detach() > cohort.B+1e-8).any():
        raise ArithmeticError("Response exceeded the budget.")
    return response


def binary_vector(values, N, name="rho"):
    values = torch.as_tensor(values).expand(N)
    if not ((values==0)|(values==1)).all():
        raise ValueError(f"{name} must be binary; convert surrogates with DefinitePolicy.")
    return values.bool()


def r(cohort, q, rho):
    rho = binary_vector(rho,cohort.N) & target_feasible(cohort,q)
    # Unselected targets are set below eligibility only for the internal solver call.
    solver_q = torch.where(rho,as_tensor(q).expand(cohort.N),torch.zeros_like(cohort.s_0))
    return br(cohort,solver_q)

## Capacity, evaluation, and baselines

`t_0` is calibrated once on reference applicants for each admission rate and is reused on validation/test applicants. Evaluation uses integer capacity, uniform cutoff tie probabilities, and averages cost and validity over **all** initial rejections, including nonrecipients and failed recourse outcomes. Target equality is checked before canonicalizing solver roundoff within a $2\times10^{-9}$ score tolerance; its residual is reported. This preserves target atoms rather than giving numerical overshoots an arbitrary tie advantage.

No-action is an actual zero-action policy. The strongest simple baseline jointly searches a common target and a cutoff on the individual cost of reaching it. All candidate targets and cost cutoffs are generated without test data; validation selects with the same $C-\lambda V$ objective.

On a finite held-out cohort, even no action can have positive validity: initial rejection uses the reference-estimated `t_0`, whereas future allocation fills that cohort's own quota. Keep this sampling effect separate from the improvement due to recourse.

In [ ]:
def initial_threshold(scores, alpha):
    scores = as_tensor(scores).reshape(-1)
    N = len(scores)
    k = math.floor(alpha * N)
    if not (0 < alpha < 1 and 1 <= k < N):
        raise ValueError("Reference capacity requires 1 <= floor(alpha*N) < N.")
    ordered = torch.sort(scores).values
    lower, upper = ordered[N - k - 1], ordered[N - k]
    if not lower < upper:
        raise ValueError("Initial boundary ties violate this experiment's continuous-score assumption.")
    t_0 = float((lower + upper) / 2)
    if not float(lower) < t_0 <= float(upper):
        t_0 = float(upper)  # Adjacent representable values: still exactly k accepted.
    return t_0


def exact_allocation(scores, k, mass=None):
    """Empirical CDF cutoff and uniform tie probabilities; also supports mixture mass."""
    scores = as_tensor(scores).detach().reshape(-1)
    mass = torch.ones_like(scores) if mass is None else as_tensor(mass).detach().reshape(-1)
    if mass.shape != scores.shape or len(scores) == 0:
        raise ValueError("Nonempty scores and matching masses are required.")
    if not torch.isfinite(scores).all() or not torch.isfinite(mass).all() or (mass < 0).any():
        raise ValueError("Invalid scores or masses.")
    total_mass = float(mass.sum())
    if not 0 <= k <= total_mass or total_mass == 0:
        raise ValueError("Capacity must lie between zero and total mass.")
    positive = mass > 0
    if k == 0:
        return torch.zeros_like(scores), float(scores[positive].max()), 0.0
    if k == total_mass:
        return torch.ones_like(scores), float(scores[positive].min()), 1.0
    ordered, indices = torch.sort(scores[positive])
    cumulative = torch.cumsum(mass[positive][indices], dim=0)
    # inf{t: F(t) >= 1-k/total_mass}; no rounding of distinct scores into ties.
    index = int(torch.searchsorted(cumulative, cumulative.new_tensor(total_mass - k)))
    t_1 = ordered[index]
    above, tied = scores > t_1, scores == t_1
    beta_r = (k - float(mass[above].sum())) / float(mass[tied].sum())
    if not -1e-12 <= beta_r <= 1 + 1e-12:
        raise ArithmeticError("Cutoff tie probability is outside [0,1].")
    beta_r = min(1.0, max(0.0, beta_r))
    y_hat_1 = above.to(scores.dtype) + beta_r * tied.to(scores.dtype)
    assert abs(float((mass * y_hat_1).sum()) - k) <= 1e-10 * max(1.0, total_mass)
    return y_hat_1, float(t_1), beta_r


@torch.no_grad()
def evaluate_policy(cohort, policy, lambda_, adoption_mask=None):
    """Deterministic recommendations; expected acceptance under uniform cutoff ties.

    The optional adoption_mask is only for the separately labelled stress test.
    t_0 is mandatory inside cohort and is never recalibrated here.
    """
    if not (np.isfinite(lambda_) and lambda_ > 0) or cohort.N_minus == 0:
        raise ValueError("Require lambda_ > 0 and at least one initially rejected applicant.")
    q, rho = policy(cohort)
    q = as_tensor(q).expand(cohort.N)
    rho = binary_vector(rho, cohort.N)
    rho = rho & target_feasible(cohort, q)
    if not torch.isfinite(q).all() or ((q[cohort.X_minus] < cohort.t_0) | (q[cohort.X_minus] > 1)).any():
        raise ValueError("Policy targets for rejected applicants must lie in [t_0,1].")
    implements = rho.clone()
    if adoption_mask is not None:
        adoption_mask = torch.as_tensor(adoption_mask)
        if adoption_mask.shape != (cohort.N,):
            raise ValueError("adoption_mask must have one entry per applicant.")
        implements &= binary_vector(adoption_mask, cohort.N, "adoption_mask")
    r_X = r(cohort, q, implements)
    costs = cost(cohort.X, r_X)
    actual_scores = cohort.f_w0(r_X)
    # Target equality is verified below before canonicalizing solver roundoff.
    # This explicit 2e-9 score tolerance preserves target ties; report its residual.
    scores = torch.where(implements, q, cohort.s_0)
    if not torch.allclose(actual_scores, scores, atol=2e-9, rtol=2e-9):
        raise ArithmeticError("The response failed its deployed-score attainment check.")
    if (costs > cohort.B + 1e-10 * max(1.0, cohort.B)).any():
        raise ArithmeticError("A response exceeded the budget.")
    assert torch.equal(r_X[~cohort.X_minus], cohort.X[~cohort.X_minus])
    assert torch.equal(r_X[:, ~cohort.actionable_mask], cohort.X[:, ~cohort.actionable_mask])
    y_hat_1, t_1, beta_r = exact_allocation(scores, cohort.k)
    C = float(costs[cohort.X_minus].mean())
    V = float(y_hat_1[cohort.X_minus].mean())
    assert V <= min(1.0,cohort.k/cohort.N_minus)+1e-12
    return dict(C=C, V=V, J=C - lambda_ * V,
                capacity_validity_bound=min(1.0,cohort.k/cohort.N_minus),
                max_target_error=float((actual_scores-scores).abs().max()),
                wasted_cost=float((costs*(1-y_hat_1))[cohort.X_minus].mean()),
                eligible_fraction=float(cohort.eligible[cohort.X_minus].double().mean()),
                globally_optimal_br=cohort.f_w0.model_type in {"affine", "quadratic"}, t_0=cohort.t_0, t_1=t_1,
                t_shift=t_1 - cohort.t_0, beta_r=beta_r,
                recommendation_rate=float(rho[cohort.X_minus].double().mean()),
                implementation_rate=float(implements[cohort.X_minus].double().mean()),
                accepted_mass=float(y_hat_1.sum()), N_minus=cohort.N_minus, k=cohort.k,
                q=q, rho=rho, r=r_X, scores=scores, y_hat_1=y_hat_1)


def scalar_metrics(result):
    return {key: value for key, value in result.items() if np.isscalar(value)}


class NoRecourse:
    def __call__(self, cohort):
        return torch.full_like(cohort.s_0, cohort.t_0), torch.zeros(cohort.N, dtype=torch.bool)


@dataclass(frozen=True)
class CommonTarget:
    q_c: float
    cost_limit: float = math.inf

    def __call__(self, cohort):
        if not cohort.t_0 <= self.q_c <= 1 or self.cost_limit < 0:
            raise ValueError("Require t_0 <= q_c <= 1 and a nonnegative cost limit.")
        q = torch.full_like(cohort.s_0, self.q_c)
        rho = target_feasible(cohort, q)
        if np.isfinite(self.cost_limit):
            rho &= cost(cohort.X, br(cohort, q)) <= self.cost_limit
        return q, rho


def select_on_validation(candidates, validation, lambda_):
    """Ties in J keep no action/earlier candidate; never use test data to select."""
    selected = NoRecourse()
    best = evaluate_policy(validation, selected, lambda_)
    for candidate in candidates:
        result = evaluate_policy(validation, candidate, lambda_)
        if result["J"] < best["J"]:
            selected, best = copy.deepcopy(candidate), result
    return selected, scalar_metrics(best)

## Smooth policy and objective

The original affine sigmoid policy surrogates are retained, using `q_cap` as their feasible target endpoint. `rho_hat` mixes the acceptance outcomes of no action and the complete response; it never interpolates features. The same `tau` appears in both the capacity root and validity surrogate. The actual nonlinear response is scored inside the training objective, and gradients pass through both the response solver and the competitive threshold.

The “all eligible” baseline fixes recommendations to one on the same eligible set while learning targets. This isolates the benefit of choosing recipients.

In [ ]:
class PolicySurrogates(nn.Module):
    """Eq. (policy-parameterization): independent affine sigmoid surrogates."""
    def __init__(self, d, Theta_1_radius=3.0, Theta_2_radius=3.0, seed=42, select_recipients=True):
        super().__init__()
        if not all(np.isfinite(value) and value > 0 for value in [Theta_1_radius, Theta_2_radius]):
            raise ValueError("Parameter-set radii must be positive.")
        generator = torch.Generator().manual_seed(seed)
        self.theta_1 = nn.Parameter(0.02 * torch.randn(d + 1, generator=generator))
        self.theta_2 = nn.Parameter(0.02 * torch.randn(d + 1, generator=generator))
        with torch.no_grad():
            self.theta_1[0] = -1.0
            self.theta_2[0] = 0.0
        self.select_recipients = select_recipients
        self.Theta_1_radius = float(Theta_1_radius)
        self.Theta_2_radius = float(Theta_2_radius)
        self.project_()

    def forward(self, cohort):
        q_hat = cohort.t_0 + (cohort.q_cap - cohort.t_0) * torch.sigmoid(cohort.X_aug @ self.theta_1)
        q_hat = torch.where(cohort.eligible, q_hat, torch.full_like(q_hat, cohort.t_0))
        rho_hat = torch.where(cohort.eligible, torch.sigmoid(cohort.X_aug @ self.theta_2),
                              torch.zeros_like(cohort.s_0))
        if not self.select_recipients:
            rho_hat = cohort.eligible.double() + 0.0*self.theta_2.sum()
        if not torch.equal(target_feasible(cohort, q_hat), cohort.eligible):
            raise FloatingPointError("Target interpolation lost feasibility at numerical precision.")
        return q_hat, rho_hat

    @torch.no_grad()
    def project_(self):
        # Theta_1 and Theta_2 are separate closed Euclidean balls.
        for parameter, radius in [(self.theta_1, self.Theta_1_radius),
                                  (self.theta_2, self.Theta_2_radius)]:
            norm = torch.linalg.vector_norm(parameter)
            if norm > radius:
                parameter.mul_(radius / norm)
        return self


class DefinitePolicy:
    def __init__(self, surrogates):
        # Clone the checkpoint: later training cannot mutate a selected policy.
        self.surrogates = copy.deepcopy(surrogates).eval()
        self.surrogates.requires_grad_(False)

    def __call__(self, cohort):
        q, rho_hat = self.surrogates(cohort)
        rho = cohort.eligible & (rho_hat >= 0.5)
        return q, rho


def a_i(t, s_0, s_br, rho_hat, tau):
    """Eq. (allocation-weights), over the entire fixed reference cohort."""
    return ((1 - rho_hat) * torch.sigmoid((s_0 - t) / tau)
            + rho_hat * torch.sigmoid((s_br - t) / tau))


def sigmoid_prime(z):
    e = torch.exp(-torch.abs(z))
    return e / (1 + e).square()


def sigmoid_difference(z_b, z_a):
    lower, upper = torch.minimum(z_a, z_b), torch.maximum(z_a, z_b)
    return (torch.sign(z_b - z_a) * torch.sigmoid(upper) * torch.sigmoid(-lower)
            * (-torch.expm1(lower - upper)))


def threshold_partials(s_0, s_br, rho_hat, t_hat, tau):
    z_0, z_br = (s_0 - t_hat) / tau, (s_br - t_hat) / tau
    derivative_0, derivative_br = sigmoid_prime(z_0), sigmoid_prime(z_br)
    denominator = ((1 - rho_hat) * derivative_0 + rho_hat * derivative_br).sum()
    if not torch.isfinite(denominator) or denominator <= torch.finfo(s_0.dtype).tiny:
        raise FloatingPointError("Threshold derivative is numerically unresolved; increase tau.")
    dt_ds_0 = (1 - rho_hat) * derivative_0 / denominator
    dt_ds_br = rho_hat * derivative_br / denominator
    dt_drho_hat = tau * sigmoid_difference(z_br, z_0) / denominator
    if not all(torch.isfinite(v).all() for v in [dt_ds_0, dt_ds_br, dt_drho_hat]):
        raise FloatingPointError("Nonfinite implicit gradient; increase tau.")
    return dt_ds_0, dt_ds_br, dt_drho_hat


class SmoothCapacity(torch.autograd.Function):
    @staticmethod
    def forward(ctx, s_0, s_br, rho_hat, k, tau):
        N = len(s_0)
        if not (1 <= k < N and isinstance(k, int) and np.isfinite(tau) and tau > 0):
            raise ValueError("Require integer 1 <= k < N and tau > 0.")
        if s_0.shape != s_br.shape or s_0.shape != rho_hat.shape:
            raise ValueError("Scores and weights must have matching shapes.")
        if not all(torch.isfinite(v).all() and ((0 <= v) & (v <= 1)).all()
                   for v in [s_0, s_br, rho_hat]):
            raise ValueError("Scores and mixture weights must lie in [0,1].")
        shift = tau * math.log((N - k) / k)
        low, high = s_0.new_tensor(shift), s_0.new_tensor(1 + shift)
        for _ in range(80):
            midpoint = (low + high) / 2
            if float(a_i(midpoint, s_0, s_br, rho_hat, tau).sum()) > k:
                low = midpoint
            else:
                high = midpoint
        t_hat = (low + high) / 2
        residual = abs(float(a_i(t_hat, s_0, s_br, rho_hat, tau).sum()) - k)
        if residual > 1e-10 * N:
            raise FloatingPointError("Bisection failed the capacity residual check.")
        ctx.save_for_backward(s_0, s_br, rho_hat, t_hat)
        ctx.tau = tau
        return t_hat

    @staticmethod
    def backward(ctx, grad_output):
        s_0, s_br, rho_hat, t_hat = ctx.saved_tensors
        derivatives = threshold_partials(s_0, s_br, rho_hat, t_hat, ctx.tau)
        return *(grad_output * derivative for derivative in derivatives), None, None




def J_hat(cohort, surrogates, tau, lambda_, mode="full"):
    """Eq. (empirical-objective); other modes are explicitly labelled ablations."""
    if (cohort.N_minus == 0 or not 1 <= cohort.k < cohort.N
            or not all(np.isfinite(value) and value > 0 for value in [tau, lambda_])):
        raise ValueError("Training requires N_minus > 0, 1 <= k < N, tau > 0, lambda_ > 0.")
    if mode not in {"full", "direct_only", "fixed_t_0"}:
        raise ValueError("Unknown gradient mode.")
    q_hat, rho_hat = surrogates(cohort)
    br_X = br(cohort, q_hat)
    # Score the actual nonlinear response and differentiate through its optimization layer.
    s_br = cohort.f_w0(br_X)
    if mode == "fixed_t_0":
        t_hat = cohort.s_0.new_tensor(cohort.t_0)
    else:
        t_hat = SmoothCapacity.apply(cohort.s_0, s_br, rho_hat, cohort.k, float(tau))
        if mode == "direct_only":
            t_hat = t_hat.detach()
    acceptance = a_i(t_hat, cohort.s_0, s_br, rho_hat, tau)
    C_hat = (rho_hat * cost(cohort.X, br_X))[cohort.X_minus].mean()
    V_hat = acceptance[cohort.X_minus].mean()
    objective = C_hat - lambda_ * V_hat
    return objective, dict(q_hat=q_hat, rho_hat=rho_hat, br=br_X, s_br=s_br,
                           t_hat=t_hat, a_i=acceptance, C_hat=C_hat, V_hat=V_hat)

## Configuration and persistent records

`experiment_config` creates an independent configuration for each dataset/model pair. Both models have their own editable entries within each experiment. Changes to one pair do not change another. Complete settings are saved in `config.json`.

The configuration ID covers scientific settings, implementation, dependency versions, and local CSV contents when applicable. Thus different fixed parameters, grids, seeds, scorer options, or response-solver options receive separate result names. Output paths, retry controls, and checkpoint frequency are operational settings and do not change this identity. Readable filename prefixes show the main parameters; the ID distinguishes settings omitted or shortened in the prefix.

Raw outputs remain in `results_dir/experiments/experiment_XX/dataset/model/config_id/`. The named aggregate in `results_dir/tables/` is refreshed after every trial. Completed result files and saved checkpoints remain available when another setting runs. All figures are written to `figures_dir` (default `./figures`), with one file per configuration and plot type. Atomic writes protect individual files. Use one running process per configuration/output directory.

Data snapshots, scorers, reachable witnesses, and matching training runs are cached. Saved results from older implementations can be exported through `export_saved_runs`, which uses the original IDs rather than trying to recompute their identities. Only reuse trusted scientific cache files.

In [ ]:
import hashlib
import inspect
import json
import os
import pickle
import platform
import re
import tempfile
import traceback
import types
from datetime import datetime, timezone
import scipy
import sklearn


def experiment_config(experiment, dataset, **overrides):
    """A fresh, fully resolved configuration. No QUICK_RUN or mutable shared defaults."""
    config = dict(
        experiment=experiment, dataset=dataset, model="quadratic", seeds=[42,43,44],
        alpha=0.4, B=0.75, lambda_=3.0, tau=0.05, T=500, eta=0.1,
        cohort_size=256, fit_cap=5000, synthetic_n=6000, gmsc_path="GiveMeSomeCredit.csv",
        methods=["No recourse","Original threshold","Common target",
                 "Common target + cost selection","Iterative common target",
                 "Personalized targets + selection","Personalized targets, all eligible"],
        lambda_values=[0.03,0.1,0.3,1,3,10,30], B_values=[0.25,0.75,1.5],
        alpha_values=[0.2,0.4,0.6], tau_values=[0.01,0.05,0.1], eta_values=[0.05,0.2],
        scorer=dict(epochs=1000,lr=0.01,hidden=12,l2_reg=0.01),
        qualification_proxy=dict(hidden=[32,16],max_iter=1500),
        solver=dict(reachability_starts=3,reachability_steps=60,response_maxiter=150,
                    response_ftol=1e-11,exact_margin=1e-4,neural_margin=0.02),
        policy=dict(Theta_1_radius=3.0,Theta_2_radius=3.0,validation_every=5),
        baselines=dict(target_points=30,cost_quantiles=10,iterative_steps=20,target_margin=0.02),
        preflight=dict(cohort_size=32,T=10,finite_difference=True),
        audit_fractions=[0.25,0.5,0.75], audit_limit=32,
        adoption_probabilities=[0,0.25,0.5,0.75,1], adoption_draws=200,
        results_dir="./Results", figures_dir="./figures", checkpoint_every=5, retry_failed=False,
    )
    if experiment == 1:
        config.update(T=500,methods=["Personalized targets + selection"])
    elif experiment == 6:
        config["methods"] += ["Fixed t_0 (ablation)","Detached threshold (ablation)"]
    elif experiment in [7,8]:
        config["methods"] = ["Personalized targets + selection"]
    for key,value in overrides.items():
        if key not in config:
            raise ValueError(f"Unknown configuration key {key!r}; check its spelling.")
        if isinstance(config[key],dict):
            if not isinstance(value,dict) or set(value)-set(config[key]):
                raise ValueError(f"Unknown or malformed options for {key}.")
            config[key].update(copy.deepcopy(value))
        else:
            config[key]=copy.deepcopy(value)
    validate_config(config)
    return config


def learned_method(method):
    choices={"Personalized targets + selection":("full",True),
             "Personalized targets, all eligible":("full",False),
             "Fixed t_0 (ablation)":("fixed_t_0",True),
             "Detached threshold (ablation)":("direct_only",True)}
    return choices.get(method)


def validate_config(c):
    datasets={"Synthetic nonlinear","Synthetic curved","German Credit","Adult",
              "ACSIncome","Give Me Some Credit"}
    if c["dataset"] not in datasets or c["model"] not in {"affine","quadratic","mlp"}:
        raise ValueError("Unknown dataset or model.")
    if c["experiment"] not in range(1,9):raise ValueError("Choose experiment 1 through 8.")
    def positive_integer(value,name,minimum=1):
        if isinstance(value,bool) or not isinstance(value,(int,np.integer)) or value<minimum:
            raise ValueError(f"{name} must be an integer >= {minimum}.")
    for name in ["T","checkpoint_every","synthetic_n","audit_limit","adoption_draws"]:
        positive_integer(c[name],name)
    positive_integer(c["cohort_size"],"cohort_size",2)
    if c["fit_cap"] is not None:positive_integer(c["fit_cap"],"fit_cap",2)
    if not c["seeds"] or len(set(c["seeds"]))!=len(c["seeds"]):
        raise ValueError("Provide distinct seeds.")
    for seed in c["seeds"]:positive_integer(seed,"seed",0)
    for key in ["lambda_","tau","eta"]:
        if not np.isfinite(c[key]) or c[key]<=0:raise ValueError(f"{key} must be positive.")
    if not np.isfinite(c["B"]) or c["B"]<0 or not 0<c["alpha"]<1:
        raise ValueError("Require B >= 0 and 0 < alpha < 1.")
    for name in ["lambda_values","B_values","alpha_values","tau_values","eta_values"]:
        values=c[name]
        if not values or len(set(values))!=len(values) or not np.isfinite(values).all():
            raise ValueError(f"{name} must contain distinct finite values.")
        if name=="B_values":valid=all(v>=0 for v in values)
        elif name=="alpha_values":valid=all(0<v<1 for v in values)
        else:valid=all(v>0 for v in values)
        if not valid:raise ValueError(f"Invalid {name}.")
    baseline_names={"No recourse","Original threshold","Common target",
                    "Common target + cost selection","Iterative common target"}
    if not c["methods"] or len(set(c["methods"]))!=len(c["methods"]):
        raise ValueError("Provide distinct methods.")
    if any(m not in baseline_names and learned_method(m) is None for m in c["methods"]):
        raise ValueError("Unknown method name.")
    if c["experiment"] in [1,8] and any(learned_method(m) is None for m in c["methods"]):
        raise ValueError("Convergence/adoption experiments require learned methods.")
    for name in ["Theta_1_radius","Theta_2_radius"]:
        if not np.isfinite(c["policy"][name]) or c["policy"][name]<=0:raise ValueError(name)
    positive_integer(c["policy"]["validation_every"],"validation_every")
    positive_integer(c["preflight"]["cohort_size"],"preflight cohort_size",2)
    positive_integer(c["preflight"]["T"],"preflight T")
    for name in ["epochs","hidden"]:positive_integer(c["scorer"][name],name)
    if c["scorer"]["lr"]<=0 or c["scorer"]["l2_reg"]<0:raise ValueError("Invalid scorer settings.")
    positive_integer(c["qualification_proxy"]["max_iter"],"qualification max_iter")
    for width in c["qualification_proxy"]["hidden"]:positive_integer(width,"qualification hidden width")
    for name in ["reachability_starts","reachability_steps","response_maxiter"]:
        positive_integer(c["solver"][name],name)
    for name in ["exact_margin","neural_margin"]:
        if not 0<c["solver"][name]<1:raise ValueError(name)
    if c["solver"]["response_ftol"]<=0:raise ValueError("response_ftol")
    for name in ["target_points","cost_quantiles","iterative_steps"]:
        positive_integer(c["baselines"][name],name,2 if name!="iterative_steps" else 1)
    if c["baselines"]["target_margin"]<=0:raise ValueError("target_margin")
    if not c["audit_fractions"] or any(not 0<v<1 for v in c["audit_fractions"]):raise ValueError("audit_fractions")
    if not c["adoption_probabilities"] or any(not 0<=v<=1 for v in c["adoption_probabilities"]):
        raise ValueError("adoption_probabilities")
    if not str(c["results_dir"]).strip():raise ValueError("Provide results_dir.")
    if not str(c.get("figures_dir", "./figures")).strip():raise ValueError("Provide figures_dir.")


def json_value(value):
    if isinstance(value,dict):return {str(k):json_value(v) for k,v in value.items()}
    if isinstance(value,(tuple,list)):return [json_value(v) for v in value]
    if isinstance(value,Path):return str(value)
    if isinstance(value,np.generic):return json_value(value.item())
    if isinstance(value,float) and not math.isfinite(value):return None
    return value


def stable_id(value):
    return hashlib.sha256(json.dumps(json_value(value),sort_keys=True,separators=(",",":"),allow_nan=False).encode()).hexdigest()[:24]


def array_id(array):
    if torch.is_tensor(array):array=array.detach().cpu().numpy()
    array=np.ascontiguousarray(array)
    return hashlib.sha256(str((array.shape,array.dtype)).encode()+array.tobytes()).hexdigest()


def slug(value):return re.sub(r"[^a-zA-Z0-9_.-]+","_",str(value)).strip("_") or "unnamed"
def utc_now():return datetime.now(timezone.utc).isoformat()


def _constant_signature(value):
    if isinstance(value,types.CodeType):return _code_signature(value)
    if isinstance(value,frozenset):return ['frozenset',sorted(_constant_signature(x) for x in value)]
    if isinstance(value,tuple):return ['tuple',[_constant_signature(x) for x in value]]
    return repr(value)


def _code_signature(code):
    # Exclude notebook cell names/line numbers so a kernel restart preserves the key.
    return [code.co_code.hex(),code.co_names,code.co_varnames,code.co_freevars,
            [_constant_signature(x) for x in code.co_consts]]


def implementation_id():
    signatures={}
    for name in IMPLEMENTATION_NAMES:
        value=globals()[name]
        members=vars(value).items() if inspect.isclass(value) else [(name,value)]
        for member,fn in members:
            if isinstance(fn,(staticmethod,classmethod)):fn=fn.__func__
            if inspect.isfunction(fn):
                fn=inspect.unwrap(fn)
                signatures[name+"."+member]=[_code_signature(fn.__code__),repr(fn.__defaults__),repr(fn.__kwdefaults__)]
    versions=dict(python=platform.python_version(),numpy=np.__version__,torch=str(torch.__version__),
                  scipy=scipy.__version__,sklearn=sklearn.__version__)
    return stable_id(dict(schema=1,code=signatures,versions=versions)),versions


def atomic_write(path,writer,binary=True):
    path=Path(path);path.parent.mkdir(parents=True,exist_ok=True)
    fd,temporary=tempfile.mkstemp(prefix=path.name+".",suffix=".tmp",dir=path.parent)
    try:
        with os.fdopen(fd,"wb" if binary else "w") as handle:
            writer(handle);handle.flush();os.fsync(handle.fileno())
        os.replace(temporary,path)
    finally:
        if os.path.exists(temporary):os.unlink(temporary)


def save_json(path,value):
    atomic_write(path,lambda f:json.dump(json_value(value),f,indent=2,sort_keys=True,allow_nan=False),binary=False)
def load_json(path):
    with Path(path).open() as f:return json.load(f)
def save_pickle(path,value):atomic_write(path,lambda f:pickle.dump(value,f,protocol=pickle.HIGHEST_PROTOCOL))
def load_pickle(path):
    # These are locally generated scientific caches; never load files from an untrusted source.
    with Path(path).open("rb") as f:return pickle.load(f)
def _tensor_payload(value):
    if isinstance(value,OrderedDict):return OrderedDict((k,_tensor_payload(v)) for k,v in value.items())
    if isinstance(value,dict):return {k:_tensor_payload(v) for k,v in value.items()}
    if isinstance(value,tuple):return tuple(_tensor_payload(v) for v in value)
    if isinstance(value,list):return [_tensor_payload(v) for v in value]
    if isinstance(value,np.generic):return value.item()
    if isinstance(value,np.ndarray):return torch.from_numpy(value.copy())
    return value


def save_torch(path,value):atomic_write(path,lambda f:torch.save(_tensor_payload(value),f))
def load_torch(path):return torch.load(path,map_location="cpu",weights_only=True)
def save_csv(path,frame):atomic_write(path,lambda f:frame.to_csv(f,index=False),binary=False)


def scientific_config(c):
    return {k:v for k,v in c.items() if k not in {"results_dir","figures_dir","retry_failed","checkpoint_every"}}


def data_source_id(config):
    if config["dataset"] != "Give Me Some Credit":
        return "fixed_loader_snapshot_v1"
    path=Path(config["gmsc_path"]).expanduser().resolve()
    digest=hashlib.sha256()
    with path.open("rb") as handle:
        for block in iter(lambda:handle.read(1024*1024),b""):digest.update(block)
    return digest.hexdigest()


def experiment_cases(c):
    axis={1:"eta",2:"lambda_",3:"B",4:"alpha",5:"tau"}.get(c["experiment"])
    grids={"eta":"eta_values","lambda_":"lambda_values","B":"B_values","alpha":"alpha_values","tau":"tau_values"}
    values=c[grids[axis]] if axis else [None]
    cases=[]
    for seed in c["seeds"]:
        for value in values:
            case=dict(seed=int(seed),alpha=float(c["alpha"]),B=float(c["B"]),lambda_=float(c["lambda_"]),
                      tau=float(c["tau"]),eta=float(c["eta"]),T=int(c["T"]))
            if axis:case[axis]=float(value)
            cases.append(case)
    return cases


def parameter_filename(config, config_id):
    """Readable key settings plus the ID of the complete scientific configuration."""
    axis = {1:'eta', 2:'lambda_', 3:'B', 4:'alpha', 5:'tau'}.get(config['experiment'])
    grids = {'eta':'eta_values', 'lambda_':'lambda_values', 'B':'B_values',
             'alpha':'alpha_values', 'tau':'tau_values'}

    def number(value):
        return f'{value:.6g}' if isinstance(value, (float, np.floating)) else str(value)

    def values_label(values):
        label = '_'.join(number(value) for value in values)
        return label if len(label) <= 20 else f'{number(values[0])}to{number(values[-1])}n{len(values)}'

    parameters = []
    for key in ['alpha', 'B', 'lambda_', 'tau', 'eta']:
        value = values_label(config[grids[key]]) if key == axis else number(config[key])
        parameters.append(f"{key.rstrip('_')}{value}")
    parameters += [f"T{config['T']}", f"N{config['cohort_size']}",
                   'seeds' + values_label(config['seeds'])]
    readable = (f"experiment-{config['experiment']}-dataset-{slug(config['dataset'])}"
                f"-model-{slug(config['model'])}-parameters-" + '-'.join(parameters))
    # Leave room for plot/table suffixes and extensions on filesystems with 255-byte names.
    return readable[:170].rstrip('-_') + '-' + config_id


def save_parameter_results(store):
    """Refresh this run's aggregate after every trial; other configurations are untouched."""
    frame = store.manifest_results()
    save_csv(store.directory / 'results.csv', frame)
    table_directory = store.root / 'tables'
    stem = parameter_filename(store.config, store.config_id)
    result_path = table_directory / f'{stem}-results.csv'
    config_path = table_directory / f'{stem}-config.json'
    save_csv(result_path, frame)
    metadata = load_json(store.directory / 'config.json')
    save_json(config_path, dict(metadata, run_directory=str(store.directory),
                                results_file=str(result_path)))
    return frame


def export_saved_runs(results_dir='./Results', *, figures_dir='./figures',
                      experiment=None, dataset=None, model=None, show=False):
    """Export completed old/new runs using their saved IDs; no fitting or dataset loading."""
    output_root = Path(results_dir).expanduser().resolve()
    exports = []
    for path in sorted((output_root / 'experiments').glob('experiment_*/*/*/*/config.json')):
        metadata = load_json(path)
        config = metadata['config']
        if experiment is not None and config['experiment'] != experiment:continue
        if dataset is not None and config['dataset'] != dataset:continue
        if model is not None and config['model'] != model:continue
        # Use the recorded identity, even if code, files, or defaults changed since training.
        store = object.__new__(RunStore)
        store.config = copy.deepcopy(config)
        store.config['figures_dir'] = str(figures_dir)
        store.config_id = metadata['config_id']
        store.code_id = metadata['implementation_id']
        store.versions = metadata.get('versions', {})
        store.source_id = metadata.get('source_id')
        store.root = output_root
        store.directory = path.parent
        if store.manifest_results().empty:continue
        frame = save_parameter_results(store)
        runner = types.SimpleNamespace(config=store.config, store=store)
        figure_paths = save_experiment_figures(runner, frame, show=show)
        stem = parameter_filename(config, store.config_id)
        exports.append(dict(experiment=config['experiment'], dataset=config['dataset'],
                            model=config['model'], config_id=store.config_id,
                            results_file=str(output_root / 'tables' / f'{stem}-results.csv'),
                            figures=figure_paths))
    return pd.DataFrame(exports)


class RunStore:
    def __init__(self,config):
        self.config=copy.deepcopy(config);validate_config(self.config)
        self.code_id,self.versions=implementation_id()
        self.source_id=data_source_id(config)
        self.config_id=stable_id(dict(config=scientific_config(config),code=self.code_id,source=self.source_id))
        self.root=Path(config["results_dir"]).expanduser().resolve()
        self.directory=self.root/"experiments"/f"experiment_{config['experiment']:02d}"/slug(config["dataset"])/config["model"]/self.config_id
        self.directory.mkdir(parents=True,exist_ok=True)
        metadata=dict(config=self.config,config_id=self.config_id,implementation_id=self.code_id,
                      source_id=self.source_id,versions=self.versions)
        existing=self.directory/"config.json"
        if existing.exists() and load_json(existing)["config_id"]!=self.config_id:
            raise RuntimeError("Configuration identity mismatch.")
        save_json(existing,metadata)

    def cache(self,kind,key):
        path=self.root/"cache"/self.code_id/kind/key;path.mkdir(parents=True,exist_ok=True);return path

    def trial(self,case,method):
        return self.directory/"trials"/f"seed_{case['seed']}"/f"{stable_id(dict(case=case,method=method))}_{slug(method)}"

    def status(self,path,status,**extra):
        save_json(Path(path)/"status.json",dict(status=status,updated_at=utc_now(),**extra))

    def manifest_results(self):
        rows=[]
        for path in sorted((self.directory/"trials").glob("seed_*/*/result.json")):
            value=load_json(path)
            if isinstance(value,list):rows.extend(value)
            else:rows.append(value)
        return pd.DataFrame(rows)

## Training and validation

Both parameter vectors take simultaneous projected full-gradient steps on fixed compact balls. The scorer, reference features, immutable mask, initial decisions, budget, and eligible set remain fixed. Validation converts `rho_hat` to binary `rho` at 0.5, includes no action, and uses the hard competitive allocation. Checkpoints are considered at the same prespecified frequency for every method. Test data never select a checkpoint, target, cost cutoff, lambda, or temperature.

Fixed-threshold and detached-threshold modes are explicit ablations. The first uses `t_0` in training; the second recomputes the competitive threshold but omits its derivative. Both are tested under real competition. The projected-gradient diagnostic is measured for the gradient actually used; the paper's stationarity guarantee applies only under its smoothness and step-size assumptions, not automatically to neural branch switches or these ablations.

`T` counts projected updates; histories contain iterations 0 through `T`. Checkpoints store both the current parameters and the best hard policy selected so far. The default checkpoint interval is five iterations. An interruption can replay work since the last checkpoint; it does not discard prior completed trials. No convergence-based early stopping or weaker solver settings have been added.

In [ ]:
def encode_policy(policy):
    if isinstance(policy,NoRecourse):return dict(type="none")
    if isinstance(policy,CommonTarget):return dict(type="common",q_c=policy.q_c,cost_limit=policy.cost_limit)
    if isinstance(policy,DefinitePolicy):
        s=policy.surrogates
        return dict(type="learned",d=s.theta_1.numel()-1,state_dict=copy.deepcopy(s.state_dict()),
                    Theta_1_radius=s.Theta_1_radius,Theta_2_radius=s.Theta_2_radius,
                    select_recipients=s.select_recipients)
    raise TypeError(type(policy).__name__)


def decode_policy(value):
    if value["type"]=="none":return NoRecourse()
    if value["type"]=="common":return CommonTarget(value["q_c"],value["cost_limit"])
    s=PolicySurrogates(value["d"],Theta_1_radius=value["Theta_1_radius"],
                       Theta_2_radius=value["Theta_2_radius"],select_recipients=value["select_recipients"])
    s.load_state_dict(value["state_dict"])
    return DefinitePolicy(s)


def learn_recourse(reference,validation,*,tau=0.05,eta=0.1,T=500,lambda_=3.0,
                   initial_surrogates=None,seed=42,mode="full",select_recipients=True,
                   validation_every=5,Theta_1_radius=3.0,Theta_2_radius=3.0,
                   checkpoint_path=None,checkpoint_key=None,checkpoint_every=5,
                   progress_callback=None):
    """Original projected full-gradient algorithm with restartable, atomic checkpoints."""
    if T<1 or eta<=0 or not np.isfinite(eta) or checkpoint_every<1:raise ValueError("Invalid training settings.")
    if reference.f_w0 is not validation.f_w0 or reference.t_0!=validation.t_0:
        raise ValueError("Reference and validation must share the scorer and initial threshold.")
    if reference.B!=validation.B or reference.alpha!=validation.alpha:raise ValueError("Cohort setting mismatch.")
    if checkpoint_path is not None and checkpoint_key is None:raise ValueError("Checkpoint identity is required.")
    surrogates=(PolicySurrogates(reference.X.shape[1],seed=seed,select_recipients=select_recipients,
                                Theta_1_radius=Theta_1_radius,Theta_2_radius=Theta_2_radius)
                if initial_surrogates is None else copy.deepcopy(initial_surrogates))
    surrogates.requires_grad_(True).project_();surrogates.select_recipients=select_recipients
    initial_state=copy.deepcopy(surrogates.state_dict())
    w_0_before={key:value.clone() for key,value in reference.f_w0.state_dict().items()}
    history=[];selected_policy=NoRecourse();selected_iteration=-1;start=0;elapsed=0.0
    best=scalar_metrics(evaluate_policy(validation,selected_policy,lambda_))
    if checkpoint_path is not None and Path(checkpoint_path).exists():
        saved=load_torch(checkpoint_path)
        if saved["checkpoint_key"]!=checkpoint_key:raise RuntimeError("Checkpoint settings do not match this run.")
        surrogates.load_state_dict(saved["state_dict"])
        initial_state=saved["initial_state"];history=saved["history"];best=saved["validation"]
        selected_policy=decode_policy(saved["selected_policy"]);selected_iteration=saved["selected_iteration"]
        start=saved["next_iteration"];elapsed=saved["elapsed_seconds"]
        if start>T+1:raise RuntimeError("Invalid checkpoint iteration.")
    resumed_from=start
    started=time.perf_counter()

    def persist(next_iteration):
        if checkpoint_path is None:return
        payload=dict(checkpoint_key=checkpoint_key,next_iteration=next_iteration,
                     state_dict=copy.deepcopy(surrogates.state_dict()),initial_state=initial_state,
                     selected_policy=encode_policy(selected_policy),selected_iteration=selected_iteration,
                     validation=best,history=history,elapsed_seconds=elapsed+time.perf_counter()-started)
        save_torch(checkpoint_path,payload)
        save_csv(Path(checkpoint_path).with_name("history.csv"),pd.DataFrame(history))

    for j in range(start,T+1):
        tick=time.perf_counter()
        surrogates.zero_grad(set_to_none=True)
        objective,outputs=J_hat(reference,surrogates,tau,lambda_,mode)
        if not torch.isfinite(objective):raise FloatingPointError("Nonfinite training objective.")
        objective.backward()
        parameters=[surrogates.theta_1,surrogates.theta_2]
        gradients=[parameter.grad.clone() for parameter in parameters]
        if not all(torch.isfinite(g).all() for g in gradients):raise FloatingPointError("Nonfinite gradient.")
        proposed=[]
        for parameter,gradient,radius in zip(parameters,gradients,[surrogates.Theta_1_radius,surrogates.Theta_2_radius]):
            candidate=parameter.detach()-eta*gradient;norm=torch.linalg.vector_norm(candidate)
            proposed.append(candidate*min(1.0,radius/float(norm)) if norm>0 else candidate)
        G_eta_sq=sum(float(((parameter.detach()-candidate)/eta).square().sum())
                     for parameter,candidate in zip(parameters,proposed))
        validation_J=float("nan")
        if j%validation_every==0 or j==T:
            definite=DefinitePolicy(surrogates)
            candidate_metrics=scalar_metrics(evaluate_policy(validation,definite,lambda_))
            validation_J=candidate_metrics["J"]
            if validation_J<best["J"]:
                selected_policy,best,selected_iteration=definite,candidate_metrics,j
        row=dict(iteration=j,J_hat=float(objective.detach()),C_hat=float(outputs["C_hat"].detach()),
                 V_hat=float(outputs["V_hat"].detach()),t_hat=float(outputs["t_hat"].detach()),G_eta_sq=G_eta_sq,
                 capacity_residual=float(outputs["a_i"].detach().sum())-reference.k,validation_J=validation_J,
                 theta_1_norm=float(surrogates.theta_1.detach().norm()),theta_2_norm=float(surrogates.theta_2.detach().norm()),
                 step_seconds=time.perf_counter()-tick)
        row["sufficient_decrease"]=(not history or row["J_hat"]<=history[-1]["J_hat"]-eta*history[-1]["G_eta_sq"]/2+1e-10)
        history.append(row)
        if j<T:
            with torch.no_grad():
                for parameter,candidate in zip(parameters,proposed):parameter.copy_(candidate)
        if j==0 or (j+1)%checkpoint_every==0 or j==T:persist(j+1)
        # Used by the runner for logging; also permits controlled interruption tests.
        if progress_callback is not None:progress_callback(j,row)
    assert all(torch.equal(value,w_0_before[key]) for key,value in reference.f_w0.state_dict().items())
    return dict(policy=selected_policy,last_iterate=copy.deepcopy(surrogates).eval(),history=pd.DataFrame(history),
                validation=best,selected_iteration=selected_iteration,initial_state=initial_state,
                resumed_from=resumed_from,runtime_seconds=elapsed+time.perf_counter()-started)

## Numerical checks and experiment runner

`run_framework_checks()` checks the response layers, gradients, capacity, immutable features, normalization, hard deployment, and splitting. It is available for an explicit diagnostic run and is not automatically repeated for each experiment.

Preflight uses the configured full fitting split and frozen scorer, but smaller reference/validation cohorts. It checks every requested seed and sweep setting on those smaller cohorts, uses the full reference threshold, probes the response and full gradient, and runs one short representative training pilot. The pilot is saved separately and never replaces a full fit. Test outcomes do not guide these checks. Its runtime estimate covers **policy training only** and is an extrapolation, not a guarantee; nonlinear solves may vary substantially between applicants. Downloading, fitting, reachability, baseline searches, and final evaluation are excluded.

Within comparisons, candidates use the same splits, scorer, immutable mask, budget, and reachable caps. Validation chooses checkpoints/common targets/cost cutoffs; test data report the final outcomes. Across-dataset collection retains configuration IDs, so unrelated settings are not silently pooled.

In [ ]:
def run_framework_checks():
    passed=[]
    # Closed-form curved example: nearest point to a circular superlevel set.
    f_w0=QuadraticScorer([1.,0.,0.,1.],[1.,1.,.2])
    X=as_tensor([[2.,0.,.5],[2.5,.1,.6],[.1,.2,.7]])
    cohort=make_cohort(X,f_w0,.65,2.,1/3,[True,True,False])
    target=float(f_w0(as_tensor([[1.,0.,.5]]))[0])
    q=torch.full((3,),target,requires_grad=True)
    response=br(cohort,q)
    assert torch.allclose(response[0],as_tensor([1.,0.,.5]),atol=1e-10)
    assert torch.equal(response[:,2],X[:,2])
    assert torch.equal(response[~cohort.X_minus],X[~cohort.X_minus])
    assert torch.autograd.gradcheck(lambda target:br(cohort,target),(q,),eps=1e-6,atol=1e-5,rtol=1e-4)
    assert torch.all(f_w0(cohort.witness)>=cohort.q_cap)
    passed.append("Global quadratic response, analytic curved example, mutable-subspace gradient")

    affine=FixedScorer([.1,1.,4.])
    sample=make_cohort([[-1.,0.],[1.,0.]],affine,.5,2.,.5,[True,False])
    q_a=torch.tensor([.6,.6])
    expected=(torch.logit(q_a[0])-.1)
    assert abs(float(br(sample,q_a)[0,0]-expected))<1e-12
    frozen=make_cohort([[-1.,0.],[1.,0.]],affine,.5,2.,.5,[False,False])
    assert not frozen.eligible.any() and torch.equal(br(frozen,q_a),frozen.X)
    zero=make_cohort([[-1.,0.],[1.,0.]],affine,.5,0.,.5,[True,False])
    assert not zero.eligible.any()
    # Immutable input is used in scoring even though response derivatives there are zero.
    assert affine(as_tensor([[0.,0.]])) != affine(as_tensor([[0.,1.]]))
    passed.append("Affine specialization; all-immutable/zero-budget cases; immutable inputs affect scores")

    class AnalyticNonlinear(nn.Module):
        model_type="mlp"
        def logits(self,X):return X[:,0]+.2*torch.sin(X[:,0])+.7*X[:,1]-.1*X[:,1].square()
        def forward(self,X):return torch.sigmoid(self.logits(X))
    nonlinear=AnalyticNonlinear().eval()
    local=make_cohort([[-1.5,.5],[-.5,-.7],[2.,0.]],nonlinear,.5,2.,1/3,[True,False])
    q_m=torch.tensor([.55,.55,.55],requires_grad=True)
    response=br(local,q_m)
    assert torch.allclose(nonlinear(response)[:2],q_m[:2],atol=2e-10)
    assert torch.equal(response[:,1],local.X[:,1])
    assert local.q_bar is None and len(local.solver_audit)>0
    assert torch.autograd.gradcheck(lambda target:br(local,target),(q_m,),eps=1e-5,atol=2e-5,rtol=3e-4)
    _,dq,_=solve_local_br(local,0,.55)
    expected_derivative=1/(.55*.45*(1+.2*torch.cos(response[0,0])))
    assert abs(float((dq[0]-expected_derivative).detach()))<1e-8
    assert all(a["kkt_residual"]<2e-7 for a in local.solver_audit)
    passed.append("Nonlinear local solver and KKT implicit derivative against an analytic inverse")

    for context in [cohort,local]:
        policy=PolicySurrogates(context.X.shape[1],seed=3)
        objective,_=J_hat(context,policy,.08,2.)
        objective.backward()
        for parameter in [policy.theta_1,policy.theta_2]:
            gradient=parameter.grad.clone()
            numerical=torch.empty_like(parameter)
            for j in range(len(parameter)):
                original=float(parameter[j].detach())
                with torch.no_grad():
                    parameter[j]=original+2e-5;high=float(J_hat(context,policy,.08,2.)[0])
                    parameter[j]=original-2e-5;low=float(J_hat(context,policy,.08,2.)[0])
                    parameter[j]=original
                numerical[j]=(high-low)/4e-5
            assert torch.allclose(gradient,numerical,atol=3e-5,rtol=1e-3)
    passed.append("Full theta_1/theta_2 gradients through nonlinear responses and competitive threshold")

    s_0=torch.tensor([.1,.3,.55,.8],requires_grad=True)
    s_br=torch.tensor([.4,.6,.7,.9],requires_grad=True)
    weights=torch.tensor([.2,.5,.7,.3],requires_grad=True)
    assert torch.autograd.gradcheck(lambda a,b,c:SmoothCapacity.apply(a,b,c,2,.07),
                                   (s_0,s_br,weights),eps=1e-6,atol=1e-6)
    t_hat=SmoothCapacity.apply(s_0,s_br,weights,2,.07)
    assert abs(float(a_i(t_hat,s_0,s_br,weights,.07).sum().detach())-2)<1e-12
    y_hat_1,_,beta=exact_allocation(torch.ones(5)*.5,2)
    assert beta==.4 and abs(float(y_hat_1.sum())-2)<1e-12
    passed.append("Integer capacity, cutoff ties, and all implicit threshold derivatives")

    class OneRecommendation:
        def __call__(self,c):return torch.full_like(c.s_0,.55),torch.tensor([True,False,False])
    result=evaluate_policy(local,OneRecommendation(),2.)
    assert abs(result["C"]-float(cost(local.X,result["r"])[0])/local.N_minus)<1e-12
    assert torch.equal(result["r"][1:],local.X[1:])
    assert torch.equal(result["r"][:,~local.actionable_mask],local.X[:,~local.actionable_mask])
    try:evaluate_policy(local,PolicySurrogates(2),2.)
    except ValueError as error:assert "binary" in str(error)
    else:raise AssertionError("Smooth recommendation weights were silently deployed.")
    fit=learn_recourse(local,local,T=2,eta=.01,lambda_=1e-8,validation_every=1)
    assert isinstance(fit["policy"],NoRecourse)
    for name in ["theta_1","theta_2"]:
        assert not torch.equal(fit["initial_state"][name],fit["last_iterate"].state_dict()[name])
    passed.append("All-rejection cost normalization, binary deployment, real no-action selection, both updates")

    rng=np.random.default_rng(7)
    raw=pd.DataFrame({"age":rng.normal(size=240),"age_bucket":rng.choice(["young","old"],240),
                      "sex":rng.choice(["F","M"],240),"income":rng.normal(size=240)})
    labels=(raw.income.to_numpy()+rng.normal(size=240)>.0).astype(int)
    pop=prepare_real_population(raw,labels,seed=7,immutable_features=["age","sex"])
    for column,mutable in zip(pop["encoded_source_features"],pop["actionable_mask"]):
        assert mutable == (column not in {"age","sex"})
    assert any(c=="age_bucket" and m for c,m in zip(pop["encoded_source_features"],pop["actionable_mask"]))
    duplicates=np.repeat(np.arange(80)[:,None],2,axis=0)
    groups=split_indices(duplicates)
    for name,ids in groups.items():
        for other,other_ids in groups.items():
            if name!=other:assert set(duplicates[ids,0]).isdisjoint(duplicates[other_ids,0])
    passed.append("Exact one-hot immutable mapping (including prefix collision) and leakage-free splits")
    return pd.DataFrame({"check":passed,"status":"passed"})

In [ ]:
def gradient_decomposition(cohort,surrogates,tau,lambda_):
    surrogates=copy.deepcopy(surrogates).requires_grad_(True)
    parameters=[surrogates.theta_1,surrogates.theta_2]
    objective,outputs=J_hat(cohort,surrogates,tau,lambda_)
    full=torch.autograd.grad(objective,parameters,retain_graph=True)
    dt=torch.autograd.grad(outputs["t_hat"],parameters)
    direct_objective,_=J_hat(cohort,surrogates,tau,lambda_,mode="direct_only")
    direct=torch.autograd.grad(direct_objective,parameters)
    with torch.no_grad():
        weights=outputs["rho_hat"]
        derivative_0=sigmoid_prime((cohort.s_0-outputs["t_hat"])/tau)
        derivative_br=sigmoid_prime((outputs["s_br"]-outputs["t_hat"])/tau)
        dJ_dt=lambda_/tau*((1-weights)*derivative_0+weights*derivative_br)[cohort.X_minus].mean()
        rows=[]
        for name,g_full,g_direct,dt_block in zip(["theta_1","theta_2"],full,direct,dt):
            assert torch.allclose(g_full-g_direct,dJ_dt*dt_block,atol=2e-8,rtol=1e-5)
            rows.append(dict(parameter=name,full_norm=float(g_full.norm()),
                             direct_norm=float(g_direct.norm()),threshold_norm=float((g_full-g_direct).norm())))
    return pd.DataFrame(rows)


@torch.no_grad()
def smoothing_diagnostic(cohort,surrogates,tau,lambda_):
    _,outputs=J_hat(cohort,surrogates,tau,lambda_)
    # Identical fractional mixture: separates acceptance smoothing from rho rounding.
    scores=torch.cat([cohort.s_0,outputs["s_br"]])
    weights=torch.cat([1-outputs["rho_hat"],outputs["rho_hat"]])
    allocation,_,_=exact_allocation(scores,cohort.k,mass=weights)
    hard=(1-outputs["rho_hat"])*allocation[:cohort.N]+outputs["rho_hat"]*allocation[cohort.N:]
    return dict(mean_acceptance_error=float((outputs["a_i"]-hard).abs().mean()),
                capacity_residual=abs(float(outputs["a_i"].sum())-cohort.k))

In [ ]:
def prepare_named_population(config, seed):
    dataset=config['dataset']
    if dataset.startswith('Synthetic'):
        X,f_star_scores,_=generate_correlated_nonlinear_population(config['synthetic_n'],correlation=0.5,seed=seed)
        if dataset=='Synthetic curved':
            f_star_scores=expit(0.6*X[:,0]+1.2*X[:,1]+0.3*X[:,2]-0.8*X[:,1]**2-0.2*X[:,2]**2)
        population=prepare_population(X,f_star_scores,seed,immutable_indices=[0])
        if config['fit_cap'] is not None and len(population['X_fit'])>config['fit_cap']:
            idx=np.random.default_rng(seed).choice(len(population['X_fit']),config['fit_cap'],replace=False)
            population['X_fit']=population['X_fit'][idx]
            population['f_star_fit']=population['f_star_fit'][idx]
            population['indices']['fit']=population['indices']['fit'][idx]
    else:
        loaders={'Adult':prepare_adult_data,'German Credit':prepare_credit_data,
                 'ACSIncome':prepare_folktables_data,'Give Me Some Credit':prepare_gmsc_data}
        options=dict(n_samples=config['fit_cap'],seed=seed,proxy_options=config['qualification_proxy'])
        if dataset=='Give Me Some Credit':options['path']=config['gmsc_path']
        population=loaders[dataset](**options)
    # Always permute a partition before taking prefixes: cohort sizes share nested applicants.
    rng=np.random.default_rng(seed+1000)
    for split in ['train','val','test']:
        indices=rng.permutation(len(population[f'X_{split}']))[:config['cohort_size']]
        population[f'X_{split}']=population[f'X_{split}'][indices]
        population['indices'][split]=population['indices'][split][indices]
        if split=='test' and 'y_test' in population:population['y_test']=population['y_test'][indices]
    return population


def scorer_payload(f_w0,config):
    return dict(model=config['model'],hidden=config['scorer']['hidden'],
                state_dict=copy.deepcopy(f_w0.state_dict()))


def restore_scorer(payload):
    state=payload['state_dict'];kind=payload['model']
    if kind=='affine':model=FixedScorer(state['w_0'])
    elif kind=='quadratic':model=QuadraticScorer(state['w_0'],state['h'])
    else:model=SmoothMLPScorer(state['w_0'],hidden=payload['hidden'])
    model.load_state_dict(state)
    return model.eval().requires_grad_(False)


def cohort_payload(cohort):
    return {key:value for key,value in vars(cohort).items() if key!='f_w0'}


def require_training_cohort(cohort):
    if cohort.N_minus==0 or not 1<=cohort.k<cohort.N:
        raise ValueError('A cohort has no initial rejections or invalid integer capacity; increase cohort_size.')


class ExperimentRunner:
    """One configuration; lazy cached data/scorer/cohorts and resumable per-method trials."""
    def __init__(self,config):
        self.store=RunStore(config);self.config=self.store.config
        self.populations={};self.scorers={};self.cohorts={};self.cohort_paths={}

    def population(self,seed):
        c=self.config
        if seed not in self.populations:
            key=stable_id(dict(dataset=c['dataset'],seed=seed,source=self.store.source_id,
                               fit_cap=c['fit_cap'],cohort_size=c['cohort_size'],synthetic_n=c['synthetic_n'],
                               qualification_proxy=c['qualification_proxy']))
            directory=self.store.cache('populations',key);path=directory/'population.pkl'
            if path.exists():population=load_pickle(path)
            else:
                print(f"Preparing {c['dataset']}, seed={seed}",flush=True)
                population=prepare_named_population(c,seed);save_pickle(path,population)
            signature=stable_id({name:array_id(value) for name,value in population.items()
                                 if name.startswith('X_') or name in ['f_star_fit','actionable_mask','y_test']})
            self.populations[seed]=(population,signature)
            save_json(directory/'metadata.json',dict(dataset=c['dataset'],seed=seed,data_id=signature,
                                                     feature_names=population['feature_names'],
                                                     immutable_features=population.get('immutable_features',['x_0'])))
        return self.populations[seed]

    def scorer(self,seed):
        if seed not in self.scorers:
            population,data_id=self.population(seed);c=self.config
            key=stable_id(dict(X_fit=array_id(population['X_fit']),f_star_fit=array_id(population['f_star_fit']),
                               seed=seed,model=c['model'],scorer=c['scorer']))
            path=self.store.cache('scorers',key)/'scorer.pt'
            if path.exists():f_w0=restore_scorer(load_torch(path))
            else:
                print(f"Fitting frozen {c['model']} scorer, seed={seed}",flush=True)
                f_w0=fit_f_w0(population['X_fit'],population['f_star_fit'],c['model'],seed,**c['scorer'])
                save_torch(path,scorer_payload(f_w0,c))
            self.scorers[seed]=(f_w0,key)
        return self.scorers[seed]

    def contexts(self,case,*,pilot=False,include_test=True):
        seed=case['seed'];population,data_id=self.population(seed);f_w0,model_id=self.scorer(seed)
        # Pilot uses the same deployed scorer and full-reference t_0, on smaller cohorts.
        t_0=initial_threshold(f_w0(as_tensor(population['X_train'])),case['alpha'])
        result={};keys={}
        for split in (['train','val','test'] if include_test else ['train','val']):
            X=population[f'X_{split}']
            if pilot:X=X[:min(len(X),self.config['preflight']['cohort_size'])]
            key=stable_id(dict(X=array_id(X),model=model_id,mask=array_id(population['actionable_mask']),
                               t_0=t_0,B=case['B'],alpha=case['alpha'],seed=seed,solver=self.config['solver']))
            if key not in self.cohorts:
                path=self.store.cache('cohorts',key)/'cohort.pt'
                if path.exists():cohort=Cohort(f_w0=f_w0,**load_torch(path))
                else:
                    cohort=make_cohort(X,f_w0,t_0,case['B'],case['alpha'],population['actionable_mask'],
                                       seed=seed,solver_options=self.config['solver'])
                    save_torch(path,cohort_payload(cohort))
                require_training_cohort(cohort)
                self.cohorts[key]=cohort;self.cohort_paths[key]=path
            result[split]=self.cohorts[key];keys[split]=key
        return result,keys

    def flush_cohorts(self,keys):
        for key in set(keys.values()):save_torch(self.cohort_paths[key],cohort_payload(self.cohorts[key]))

    def fit(self,case,contexts,keys,method,*,pilot=False,progress_callback=None,trial_directory=None):
        mode,select=learned_method(method)
        options=dict(tau=case['tau'],eta=case['eta'],T=case['T'],lambda_=case['lambda_'],seed=case['seed'],
                     mode=mode,select_recipients=select,**self.config['policy'])
        key=stable_id(dict(train=keys['train'],val=keys['val'],options=options,pilot=pilot))
        path=self.store.cache('pilot_fits' if pilot else 'fits',key)
        if trial_directory is not None:
            save_json(Path(trial_directory)/'fit.json',dict(fit_id=key,fit_path=str(path),
                        checkpoint_path=str(path/'checkpoint.pt'),history_path=str(path/'history.csv')))
        fit=learn_recourse(contexts['train'],contexts['val'],**options,
                           checkpoint_path=path/'checkpoint.pt',checkpoint_key=key,
                           checkpoint_every=self.config['checkpoint_every'],progress_callback=progress_callback)
        save_torch(path/'policy.pt',encode_policy(fit['policy']))
        save_json(path/'summary.json',dict(selected_iteration=fit['selected_iteration'],
                                           validation=fit['validation'],runtime_seconds=fit['runtime_seconds']))
        fit['fit_path']=str(path);fit['fit_id']=key
        return fit

    def baseline(self,case,contexts,keys,method):
        reference,validation=contexts['train'],contexts['val']
        if method=='No recourse':return NoRecourse()
        if method=='Original threshold':return CommonTarget(reference.t_0)
        settings=self.config['baselines']
        key=stable_id(dict(train=keys['train'],val=keys['val'],method=method,settings=settings))
        directory=self.store.cache('baseline_searches',key);state_path=directory/'search.pt'
        state=load_torch(state_path) if state_path.exists() else dict(next_target=0,entries=[],q_c=reference.t_0)
        # Save each completed target block. Interrupted grid searches resume here.
        count=settings['iterative_steps'] if method=='Iterative common target' else settings['target_points']
        grid=np.linspace(reference.t_0,1.0,settings['target_points'])
        for index in range(state['next_target'],count):
            q_c=state['q_c'] if method=='Iterative common target' else float(grid[index])
            candidate=CommonTarget(q_c);candidates=[candidate]
            if method=='Common target + cost selection':
                q,rho=candidate(reference)
                if rho.any():
                    costs=cost(reference.X,br(reference,q))[rho]
                    limits=torch.quantile(costs,torch.linspace(0,1,settings['cost_quantiles']))
                    candidates += [CommonTarget(q_c,float(limit)) for limit in torch.unique(limits)]
            entries=[]
            for policy in candidates:
                metrics=scalar_metrics(evaluate_policy(validation,policy,1.0))
                entries.append(dict(policy=encode_policy(policy),C=metrics['C'],V=metrics['V']))
            if method=='Iterative common target':
                result=evaluate_policy(reference,candidate,1.0)
                state['q_c']=min(1.,max(q_c,result['t_1'])+settings['target_margin'])
            state['entries'].extend(entries);state['next_target']=index+1
            save_torch(state_path,state)
        selected=NoRecourse();best=evaluate_policy(validation,selected,case['lambda_'])['J']
        for entry in state['entries']:
            value=entry['C']-case['lambda_']*entry['V']
            if value<best:selected,best=decode_policy(entry['policy']),value
        return selected

    def tags(self,case,method):
        return dict(experiment=self.config['experiment'],dataset=self.config['dataset'],model=self.config['model'],
                    config_id=self.store.config_id,implementation_id=self.store.code_id,method=method,**case)

    def run_trial(self,case,method):
        directory=self.store.trial(case,method);result_path=directory/'result.json'
        if result_path.exists():
            status_path=directory/'status.json'
            if not status_path.exists() or load_json(status_path)['status']!='completed':
                self.store.status(directory,'completed',case=case,method=method)
            print(f"  Saved: {method}, seed={case['seed']}",flush=True);return
        status_path=directory/'status.json'
        if status_path.exists() and load_json(status_path)['status']=='failed' and not self.config['retry_failed']:
            print(f"  Prior failure: {method}; set retry_failed=True after checking error.txt.",flush=True);return
        self.store.status(directory,'running',case=case,method=method)
        keys={};started=time.perf_counter()
        try:
            contexts,keys=self.contexts(case);fit=None
            print(f"  Running {method}: seed={case['seed']}, lambda={case['lambda_']:g}, "
                  f"B={case['B']:g}, alpha={case['alpha']:g}, tau={case['tau']:g}, eta={case['eta']:g}",flush=True)
            if learned_method(method) is not None:
                def progress(j,row):
                    if j%max(100,self.config['checkpoint_every'])==0 or j==case['T']:
                        print(f"    {j}/{case['T']}: J_hat={row['J_hat']:.5g}, G_eta_sq={row['G_eta_sq']:.3g}",flush=True)
                fit=self.fit(case,contexts,keys,method,progress_callback=progress,trial_directory=directory)
                policy=fit['policy'];save_csv(directory/'history.csv',fit['history'])
                save_json(directory/'fit.json',dict(fit_id=fit['fit_id'],fit_path=fit['fit_path'],
                                                    checkpoint_path=str(Path(fit['fit_path'])/'checkpoint.pt'),
                                                    history_path=str(Path(fit['fit_path'])/'history.csv'),
                                                    selected_iteration=fit['selected_iteration'],
                                                    runtime_seconds=fit['runtime_seconds']))
            else:policy=self.baseline(case,contexts,keys,method)
            save_torch(directory/'policy.pt',encode_policy(policy))
            validation=evaluate_policy(contexts['val'],policy,case['lambda_'])
            metrics=evaluate_policy(contexts['test'],policy,case['lambda_'])
            row=dict(**self.tags(case,method),status='ok',expected_methods=len(self.config['methods']),
                     validation_J=validation['J'],trial_seconds=time.perf_counter()-started,**scalar_metrics(metrics))
            if fit is not None:
                row.update(selected_iteration=fit['selected_iteration'],training_seconds=fit['runtime_seconds'])
                if self.config['experiment']==5:
                    row.update(smoothing_diagnostic(contexts['train'],fit['last_iterate'],case['tau'],case['lambda_']))
                if self.config['experiment']==6 and method=='Personalized targets + selection':
                    save_csv(directory/'gradient_decomposition.csv',gradient_decomposition(
                        contexts['train'],fit['last_iterate'],case['tau'],case['lambda_']))
            if self.config['experiment']==8:
                self.adoption(case,method,contexts['test'],policy,directory,row)
            else:save_json(result_path,row)
            self.store.status(directory,'completed',case=case,method=method)
        except KeyboardInterrupt:
            self.store.status(directory,'interrupted',case=case,method=method)
            print('Interrupted. Completed trials and the last training checkpoint are saved.',flush=True)
            raise
        except Exception as error:
            atomic_write(directory/'error.txt',lambda f:f.write(traceback.format_exc()),binary=False)
            self.store.status(directory,'failed',case=case,method=method,error=str(error))
            print(f"  Failed (saved): {type(error).__name__}: {error}",flush=True)
        finally:
            self.flush_cohorts(keys)
            save_parameter_results(self.store)

    def adoption(self,case,method,cohort,policy,directory,base_row):
        rng=np.random.default_rng(case['seed'])
        uniforms=rng.uniform(size=(self.config['adoption_draws'],cohort.N))
        rows=[]
        for p_adopt in self.config['adoption_probabilities']:
            path=directory/'adoption'/f'{stable_id(p_adopt)}.json'
            if path.exists():part=load_json(path)
            else:
                part=[]
                for draw,uniform in enumerate(uniforms):
                    metrics=scalar_metrics(evaluate_policy(cohort,policy,case['lambda_'],adoption_mask=uniform<p_adopt))
                    part.append(dict(base_row,**dict(metrics,p_adopt=p_adopt,draw=draw)))
                save_json(path,part)
            rows.extend(part)
        save_json(directory/'result.json',rows)

    def audit(self,case):
        method='Solver and model audit';directory=self.store.trial(case,method)
        if (directory/'result.json').exists():return
        old=directory/'status.json'
        if old.exists() and load_json(old)['status']=='failed' and not self.config['retry_failed']:return
        keys={};self.store.status(directory,'running',case=case,method=method)
        try:
            contexts,keys=self.contexts(case);cohort=contexts['train'];records=[]
            selected=torch.where(cohort.eligible)[0][:self.config['audit_limit']]
            for fraction in self.config['audit_fractions']:
                q=cohort.s_0.clone();q[selected]=cohort.t_0+fraction*(cohort.q_cap[selected]-cohort.t_0)
                # Only audit selected rejected applicants; other q entries are deliberately infeasible/no-action.
                with torch.no_grad():responses=br(cohort,q)
                for i in selected.tolist():
                    records.append(dict(applicant=i,target_fraction=fraction,
                                        target_error=abs(float(cohort.f_w0(responses[i:i+1])[0])-float(q[i])),
                                        cost=float(cost(cohort.X[i:i+1],responses[i:i+1])[0]),
                                        immutable_error=float((responses[i,~cohort.actionable_mask]-cohort.X[i,~cohort.actionable_mask]).abs().max())
                                            if (~cohort.actionable_mask).any() else 0.))
            save_csv(directory/'response_probes.csv',pd.DataFrame(records))
            save_csv(directory/'solver_audit.csv',pd.DataFrame(cohort.solver_audit))
            population,_=self.population(case['seed']);audit=cohort.solver_audit
            row=dict(**self.tags(case,method),status='ok',expected_methods=1,n_probes=len(records),
                     eligible_fraction=float(cohort.eligible[cohort.X_minus].double().mean()),
                     reachability_exact=cohort.q_bar is not None,globally_optimal_br=cohort.q_bar is not None,
                     max_target_error=max([r['target_error'] for r in records],default=0.),
                     max_immutable_error=max([r['immutable_error'] for r in records],default=0.),
                     max_kkt_residual=max([r['kkt_residual'] for r in audit],default=0.),
                     max_local_cost_spread=max([r['local_cost_spread'] for r in audit],default=0.),
                     immutable_features=int((~cohort.actionable_mask).sum()))
            if 'y_test' in population:
                scores=contexts['test'].s_0.numpy();y=population['y_test']
                row.update(predictive_log_loss=log_loss(y,scores,labels=[0,1]),
                           predictive_auc=roc_auc_score(y,scores) if len(np.unique(y))==2 else None)
            save_json(directory/'result.json',row);self.store.status(directory,'completed')
        except KeyboardInterrupt:
            self.store.status(directory,'interrupted');raise
        except Exception as error:
            atomic_write(directory/'error.txt',lambda f:f.write(traceback.format_exc()),binary=False)
            self.store.status(directory,'failed',error=str(error));print(f'Audit failed (saved): {error}')
        finally:
            self.flush_cohorts(keys);save_parameter_results(self.store)

In [ ]:
def directional_gradient_check(cohort,surrogates,tau,lambda_):
    surrogates=copy.deepcopy(surrogates).requires_grad_(True)
    objective,_=J_hat(cohort,surrogates,tau,lambda_)
    parameters=[surrogates.theta_1,surrogates.theta_2]
    gradients=torch.autograd.grad(objective,parameters)
    generator=torch.Generator().manual_seed(91)
    directions=[torch.randn(p.shape,generator=generator) for p in parameters]
    scale=sum(float(d.square().sum()) for d in directions)**0.5
    directions=[d/scale for d in directions];epsilon=1e-5
    analytic=sum(float((g*d).sum()) for g,d in zip(gradients,directions))
    values=[]
    for sign in [-1,1]:
        candidate=copy.deepcopy(surrogates)
        with torch.no_grad():
            for p,d in zip([candidate.theta_1,candidate.theta_2],directions):p.add_(sign*epsilon*d)
            values.append(float(J_hat(cohort,candidate,tau,lambda_)[0]))
    numerical=(values[1]-values[0])/(2*epsilon)
    error=abs(analytic-numerical)
    if not np.isfinite(error) or error>2e-5+2e-3*max(abs(analytic),abs(numerical)):
        raise ArithmeticError(f'Directional gradient check failed: analytic={analytic}, finite difference={numerical}.')
    return dict(analytic_derivative=analytic,finite_difference_derivative=numerical,absolute_error=error)


def preflight(runner):
    c=runner.config;directory=runner.store.directory/'preflight';path=directory/'report.json'
    if path.exists() and load_json(path).get('status')=='passed':
        report=load_json(path);print('Loaded saved preflight.',flush=True);return report
    started=time.perf_counter();rows=[];warnings_list=[];keys_seen={}
    runner.store.status(directory,'running')
    try:
        cases=experiment_cases(c);first_contexts=None;first_keys=None
        for index,case in enumerate(cases):
            contexts,keys=runner.contexts(case,pilot=True,include_test=False);keys_seen.update({v:v for v in keys.values()})
            if first_contexts is None:first_contexts,first_keys=contexts,keys
            surrogates=PolicySurrogates(contexts['train'].X.shape[1],seed=case['seed'],
                                       Theta_1_radius=c['policy']['Theta_1_radius'],Theta_2_radius=c['policy']['Theta_2_radius'])
            objective,outputs=J_hat(contexts['train'],surrogates,case['tau'],case['lambda_'])
            gradients=torch.autograd.grad(objective,[surrogates.theta_1,surrogates.theta_2])
            if not torch.isfinite(objective) or not all(torch.isfinite(g).all() for g in gradients):
                raise FloatingPointError('Preflight found a nonfinite objective/gradient.')
            for split,cohort in contexts.items():
                result=evaluate_policy(cohort,DefinitePolicy(surrogates),case['lambda_'])
                quantiles=torch.quantile(cohort.s_0,as_tensor([0.,0.5,1.]))
                rows.append(dict(case,split=split,N=cohort.N,N_minus=cohort.N_minus,k=cohort.k,t_0=cohort.t_0,
                                 eligible=int(cohort.eligible.sum()),eligible_fraction=result['eligible_fraction'],
                                 min_score=float(quantiles[0]),median_score=float(quantiles[1]),max_score=float(quantiles[2]),
                                 max_reachable_improvement=float((cohort.q_cap-cohort.s_0).max()),
                                 immutable_features=int((~cohort.actionable_mask).sum()),
                                 recommendation_rate=result['recommendation_rate'],max_target_error=result['max_target_error'],
                                 gradient_norm=float(sum(g.square().sum() for g in gradients).sqrt()) if split=='train' else None))
                if result['eligible_fraction']<0.05:
                    warnings_list.append(f"seed={case['seed']}, B={case['B']}, alpha={case['alpha']}, {split}: fewer than 5% eligible.")
            save_csv(directory/'diagnostics.csv',pd.DataFrame(rows))
        gradient=None
        if c['preflight']['finite_difference']:
            gradient=directional_gradient_check(first_contexts['train'],PolicySurrogates(
                first_contexts['train'].X.shape[1],seed=cases[0]['seed'],
                Theta_1_radius=c['policy']['Theta_1_radius'],Theta_2_radius=c['policy']['Theta_2_radius']),
                cases[0]['tau'],cases[0]['lambda_'])
        fit=None;training_count=0;estimate=0.;step_seconds=0.
        if c['experiment']!=7:
            pilot_case=dict(cases[0],T=min(c['T'],c['preflight']['T']))
            method=next((m for m in c['methods'] if learned_method(m)),None)
            if method is not None:
                fit=runner.fit(pilot_case,first_contexts,first_keys,method,pilot=True)
                save_csv(directory/'pilot_history.csv',fit['history'])
                step_seconds=float(fit['history'].step_seconds.median())
                training_count=len(cases)*sum(learned_method(m) is not None for m in c['methods'])
                population,_=runner.population(cases[0]['seed'])
                ratio=len(population['X_train'])/first_contexts['train'].N
                estimate=training_count*(c['T']+1)*step_seconds*ratio
                if float(fit['history'].G_eta_sq.max())<1e-14:
                    warnings_list.append('Pilot gradients are near zero; inspect eligibility, tau, and the objective before a full run.')
                if float(fit['history'].sufficient_decrease.mean())<0.8:
                    warnings_list.append('Frequent pilot descent-check failures; inspect the history and consider a smaller eta.')
        report=dict(status='passed',config_id=runner.store.config_id,created_at=utc_now(),
                    planned_cases=len(cases),planned_trials=len(cases)*(1 if c['experiment']==7 else len(c['methods'])),
                    planned_training_runs=training_count,pilot_seconds_per_iteration=step_seconds,
                    rough_uncached_training_hours=estimate/3600,
                    rough_training_hours_range=[0.5*estimate/3600,2*estimate/3600],
                    timing_note='A rough extrapolation from one small pilot, not a runtime bound. Excludes data/scorer setup, '
                                'reachability, baseline searches, test evaluation and plotting; nonlinear solver difficulty can vary.',
                    gradient_check=gradient,warnings=sorted(set(warnings_list)),elapsed_seconds=time.perf_counter()-started)
        save_json(path,report);runner.store.status(directory,'passed');return report
    except KeyboardInterrupt:
        runner.store.status(directory,'interrupted');raise
    except Exception as error:
        atomic_write(directory/'error.txt',lambda f:f.write(traceback.format_exc()),binary=False)
        save_json(path,dict(status='failed',error=str(error),config_id=runner.store.config_id))
        runner.store.status(directory,'failed',error=str(error));raise
    finally:runner.flush_cohorts(keys_seen)


def load_histories(store):
    frames=[]
    for path in sorted((store.directory/'trials').glob('seed_*/*/history.csv')):
        status_path=path.parent/'status.json'
        if not status_path.exists():continue
        metadata=load_json(status_path)
        if 'case' not in metadata:continue
        history=pd.read_csv(path)
        for key,value in metadata['case'].items():history[key]=value
        history['method']=metadata['method'];history['trial_id']=path.parent.name
        frames.append(history)
    return pd.concat(frames,ignore_index=True) if frames else pd.DataFrame()


def complete_comparisons(frame):
    if frame.empty or 'expected_methods' not in frame:return frame
    keys=['config_id','seed','alpha','B','lambda_','tau','eta','T']
    count=frame.groupby(keys)['method'].transform('nunique')
    keep=count==frame.expected_methods
    if not keep.all():print('Plots omit incomplete method comparisons. All raw successes and failure records remain saved.')
    return frame[keep].copy()


def save_experiment_figures(runner,results,*,show=True):
    c=runner.config;e=c['experiment']
    directory=Path(c.get('figures_dir','./figures')).expanduser().resolve()
    directory.mkdir(parents=True,exist_ok=True)
    stem=parameter_filename(c,runner.store.config_id)
    figures=[];frame=complete_comparisons(results)
    if e==1:
        histories=load_histories(runner.store)
        if not histories.empty:
            fig,axes=plt.subplots(1,3,figsize=(12,3.6))
            for (method,eta),group in histories.groupby(['method','eta']):
                group=group.sort_values(['seed','iteration']).copy()
                group['best_G']=group.groupby('seed').G_eta_sq.cummin()
                group['residual']=group.capacity_residual.abs()
                mean=group.groupby('iteration')[['J_hat','best_G','residual']].mean()
                for ax,key in zip(axes,['J_hat','best_G','residual']):ax.plot(mean.index,mean[key],label=f'{method}; eta={eta:g}')
            for ax,label in zip(axes,['Training objective','Best squared gradient mapping','Capacity residual']):
                ax.set(xlabel='Projected updates',ylabel=label);ax.grid(alpha=.25)
            axes[1].set_yscale('symlog',linthresh=1e-12)
            positive=histories.loc[histories.G_eta_sq>0,'G_eta_sq']
            if len(positive):
                lower=10.**math.floor(math.log10(float(positive.min())))
                upper=10.**math.ceil(math.log10(float(positive.max())))
                axes[1].set_ylim(0 if (histories.G_eta_sq==0).any() else lower,
                                 upper if upper>lower else 10*upper)
            axes[0].ticklabel_format(axis='y',style='plain',useOffset=False)
            axes[0].legend(fontsize=6);figures.append(('convergence',fig))
    elif e==2 and not frame.empty:
        fig,ax=plt.subplots(figsize=(7,4.8))
        for method,group in frame.groupby('method',sort=False):
            aggregate=group.groupby('lambda_')[['C','V']];mean=aggregate.mean();se=aggregate.sem().fillna(0)
            ax.errorbar(mean.V,mean.C,xerr=se.V,yerr=se.C,marker='o',capsize=2,label=method)
            if method=='Personalized targets + selection':
                for lam,row in mean.iterrows():ax.annotate(f'{lam:g}',(row.V,row.C),fontsize=7)
        ax.set(xlabel='Post-shift validity V',ylabel='Mean recourse cost C');ax.grid(alpha=.25);ax.legend(fontsize=7)
        figures.append(('cost_validity',fig))
    elif e in [3,4,5] and not frame.empty:
        axis={3:'B',4:'alpha',5:'tau'}[e];fig,axes=plt.subplots(1,3,figsize=(12,3.6))
        for method,group in frame.groupby('method',sort=False):
            aggregate=group.groupby(axis)[['C','V','J']];mean=aggregate.mean();se=aggregate.sem().fillna(0)
            for ax,key in zip(axes,['C','V','J']):
                ax.errorbar(mean.index,mean[key],yerr=se[key],marker='o',capsize=2,label=method)
                ax.set(xlabel=axis,ylabel=key);ax.grid(alpha=.25)
        axes[0].legend(fontsize=5);figures.append((f'{axis}_sensitivity',fig))
    elif e==6 and not frame.empty:
        fig,axes=plt.subplots(1,3,figsize=(14,4.5));aggregate=frame.groupby('method')[['C','V','J']]
        mean=aggregate.mean();se=aggregate.sem().fillna(0)
        for ax,key in zip(axes,['C','V','J']):
            ax.barh(mean.index,mean[key],xerr=se[key]);ax.set_xlabel(key)
        figures.append(('baselines_and_ablations',fig))
    elif e==8 and not frame.empty:
        # Average Monte Carlo draws within each seed before standard errors across seeds.
        seed_means=frame.groupby(['method','seed','p_adopt'])[['C','V']].mean().reset_index()
        fig,axes=plt.subplots(1,2,figsize=(9,3.6))
        for method,group in seed_means.groupby('method'):
            mean=group.groupby('p_adopt')[['C','V']].mean();se=group.groupby('p_adopt')[['C','V']].sem().fillna(0)
            for ax,key in zip(axes,['C','V']):
                ax.errorbar(mean.index,mean[key],yerr=se[key],marker='o',label=method);ax.set(xlabel='Adoption probability',ylabel=key)
        axes[0].legend(fontsize=7);figures.append(('adoption',fig))
    for name,fig in figures:
        fig.suptitle(f"{c['dataset']} | {c['model']}");fig.tight_layout()
        path=directory/f'{stem}-{name}.png'
        atomic_write(path,lambda handle:fig.savefig(handle,format='png',dpi=160,bbox_inches='tight'))
        save_json(directory/f'{stem}-{name}.json',dict(
            config=c,config_id=runner.store.config_id,plot=name,
            run_directory=str(runner.store.directory),figure_file=str(path)))
        if show:plt.show()
        plt.close(fig)
    return [str(directory/f'{stem}-{name}.png') for name,_ in figures]


def select_experiment_configs(configs_by_dataset, selected_datasets, selected_models=None):
    """Dataset order first, then model order; returned configurations are independent copies."""
    if isinstance(selected_datasets, str):
        raise ValueError('Use a list of dataset names, for example ["German Credit"].')
    names = list(selected_datasets)
    if len(names) != len(set(names)):
        raise ValueError('Select each dataset only once.')
    unknown = [name for name in names if name not in configs_by_dataset]
    if unknown:
        raise ValueError(f'Unknown selected datasets: {unknown}. Available: {list(configs_by_dataset)}')
    if isinstance(selected_models, str):
        raise ValueError('Use a list of models, for example ["affine", "quadratic"].')
    models = None if selected_models is None else list(selected_models)
    if models is not None and len(models) != len(set(models)):
        raise ValueError('Select each model only once.')
    selected = []
    for dataset in names:
        entry = configs_by_dataset[dataset]
        # Older flat configuration dictionaries remain accepted for programmatic callers.
        by_model = {entry['model']:entry} if 'model' in entry else entry
        requested = list(by_model) if models is None else models
        missing = [name for name in requested if name not in by_model]
        if missing:
            raise ValueError(f'{dataset}: models {missing} are not configured. Available: {list(by_model)}')
        for model in requested:
            config = copy.deepcopy(by_model[model])
            if config['dataset'] != dataset or config['model'] != model:
                raise ValueError('The configuration dataset/model must match its dictionary keys.')
            selected.append(config)
    return selected


def execute_experiment(configs,*,mode='run',show=True):
    """Run selected configurations sequentially; each keeps its own saved results."""
    if mode not in {'preflight','run','load','skip'}:raise ValueError('Unknown execution mode.')
    if mode=='skip':return []
    configs=list(configs)
    if not configs:
        print('No datasets selected; nothing to run.')
        return []
    for config in configs:validate_config(config)
    reports=[]
    for position,config in enumerate(configs,1):
        print(f"Selected configuration {position}/{len(configs)}: {config['dataset']}",flush=True)
        runner=ExperimentRunner(config);c=runner.config
        print(f"Experiment {c['experiment']} | {c['dataset']} | {c['model']} | {runner.store.config_id}")
        print(f'Results: {runner.store.directory}',flush=True)
        print(f"Named tables: {runner.store.root / 'tables'} | Figures: {Path(c.get('figures_dir','./figures')).resolve()}",flush=True)
        if mode!='load':
            report=preflight(runner);reports.append(report)
            display(pd.DataFrame([{key:value for key,value in report.items()
                                   if key in ['status','planned_trials','planned_training_runs','pilot_seconds_per_iteration','rough_uncached_training_hours']}]))
            for message in report['warnings']:print('Preflight note:',message)
            print(report['timing_note'])
            if mode=='preflight':
                display(pd.read_csv(runner.store.directory/'preflight'/'diagnostics.csv'))
                print('Review these diagnostics; set this experiment mode to "run" to start its full configuration.')
                continue
            for case in experiment_cases(c):
                if c['experiment']==7:runner.audit(case)
                else:
                    for method in c['methods']:runner.run_trial(case,method)
        frame=save_parameter_results(runner.store)
        failures=[]
        for path in sorted((runner.store.directory/'trials').glob('seed_*/*/status.json')):
            record=load_json(path)
            if record['status']!='completed':failures.append(dict(trial_directory=str(path.parent),**record))
        save_json(runner.store.directory/'incomplete_trials.json',failures)
        if failures:print(f'{len(failures)} incomplete/failed trials; see incomplete_trials.json. No fabricated zero results were added.')
        if not frame.empty:display(frame);save_experiment_figures(runner,frame,show=show)
        else:print('No saved successful trials for this configuration.')
        reports.append(dict(directory=str(runner.store.directory),results=frame,incomplete=failures))
    return reports


def collect_saved_results(results_dir='./Results/',*,experiment=None,dataset=None):
    """Collect all saved datasets/configurations without fitting models or rerunning trials."""
    root=Path(results_dir).expanduser().resolve();rows=[]
    for path in sorted((root/'experiments').glob('experiment_*/*/*/*/trials/seed_*/*/result.json')):
        value=load_json(path);entries=value if isinstance(value,list) else [value]
        for row in entries:
            if experiment is not None and row['experiment']!=experiment:continue
            if dataset is not None and row['dataset']!=dataset:continue
            rows.append(row)
    frame=pd.DataFrame(rows)
    suffix=slug(f"experiment_{experiment or 'all'}_{dataset or 'all_datasets'}")
    save_csv(root/'summaries'/f'{suffix}.csv',frame)
    return frame

In [ ]:
IMPLEMENTATION_NAMES = tuple(sorted(
    name for name, value in list(globals().items())
    if (inspect.isfunction(value) or inspect.isclass(value))
    and getattr(value, "__module__", None) == __name__
    and name not in {"_code_signature", "implementation_id"}
))
print("Common setup ready. Run the configuration and execution cells for your chosen experiment.")
# Optional diagnostic: display(run_framework_checks())

## Experiment 1: Convergence

Compare step sizes `eta_values=[0.05, 0.2]` separately at `lambda_values=[3, 30]`, using `tau=0.01`, `B=0.75`, and `alpha=0.4`. This checks both the cost-sensitive and validity-sensitive regimes. Save the smooth objective, squared projected-gradient mapping, capacity residual, and sufficient-decrease diagnostics over 500 updates. A decreasing objective alone does not establish stationarity.

All six datasets have independent affine/quadratic configurations below. Choose datasets and models with the two selection lists.


In [ ]:
# Independent configurations: edit any dataset/model pair without affecting the others.
# lambda_values is the executed lambda grid (experiment 7 does not sweep lambda).
# lambda_ is a scalar base value; execution below expands the listed lambda values.
experiment_1_configs = {
    'German Credit': {
        'affine': experiment_config(1, 'German Credit', model='affine',
            seeds=[42, 43, 44, 45, 46], alpha=0.4, B=0.75, lambda_=30.0,
            tau=0.01, eta=0.2, T=500,
            cohort_size=256, fit_cap=5000, synthetic_n=6000,
            lambda_values=[3.0, 30.0],
            eta_values=[0.05, 0.2],
            baselines=dict(target_points=300, cost_quantiles=50),
            results_dir="./Results", figures_dir="./figures"),
        'quadratic': experiment_config(1, 'German Credit', model='quadratic',
            seeds=[42, 43, 44, 45, 46], alpha=0.4, B=0.75, lambda_=30.0,
            tau=0.01, eta=0.2, T=500,
            cohort_size=256, fit_cap=5000, synthetic_n=6000,
            lambda_values=[3.0, 30.0],
            eta_values=[0.05, 0.2],
            baselines=dict(target_points=300, cost_quantiles=50),
            results_dir="./Results", figures_dir="./figures"),
    },

    'Adult': {
        'affine': experiment_config(1, 'Adult', model='affine',
            seeds=[42, 43, 44, 45, 46], alpha=0.4, B=0.75, lambda_=30.0,
            tau=0.01, eta=0.2, T=500,
            cohort_size=256, fit_cap=5000, synthetic_n=6000,
            lambda_values=[3.0, 30.0],
            eta_values=[0.05, 0.2],
            baselines=dict(target_points=300, cost_quantiles=50),
            results_dir="./Results", figures_dir="./figures"),
        'quadratic': experiment_config(1, 'Adult', model='quadratic',
            seeds=[42, 43, 44, 45, 46], alpha=0.4, B=0.75, lambda_=30.0,
            tau=0.01, eta=0.2, T=500,
            cohort_size=256, fit_cap=5000, synthetic_n=6000,
            lambda_values=[3.0, 30.0],
            eta_values=[0.05, 0.2],
            baselines=dict(target_points=300, cost_quantiles=50),
            results_dir="./Results", figures_dir="./figures"),
    },

    'ACSIncome': {
        'affine': experiment_config(1, 'ACSIncome', model='affine',
            seeds=[42, 43, 44, 45, 46], alpha=0.4, B=0.75, lambda_=30.0,
            tau=0.01, eta=0.2, T=500,
            cohort_size=256, fit_cap=5000, synthetic_n=6000,
            lambda_values=[3.0, 30.0],
            eta_values=[0.05, 0.2],
            baselines=dict(target_points=300, cost_quantiles=50),
            results_dir="./Results", figures_dir="./figures"),
        'quadratic': experiment_config(1, 'ACSIncome', model='quadratic',
            seeds=[42, 43, 44, 45, 46], alpha=0.4, B=0.75, lambda_=30.0,
            tau=0.01, eta=0.2, T=500,
            cohort_size=256, fit_cap=5000, synthetic_n=6000,
            lambda_values=[3.0, 30.0],
            eta_values=[0.05, 0.2],
            baselines=dict(target_points=300, cost_quantiles=50),
            results_dir="./Results", figures_dir="./figures"),
    },

    'Give Me Some Credit': {
        'affine': experiment_config(1, 'Give Me Some Credit', model='affine',
            seeds=[42, 43, 44, 45, 46], alpha=0.4, B=0.75, lambda_=30.0,
            tau=0.01, eta=0.2, T=500,
            cohort_size=256, fit_cap=5000, synthetic_n=6000,
            lambda_values=[3.0, 30.0],
            eta_values=[0.05, 0.2],
            baselines=dict(target_points=300, cost_quantiles=50),
            gmsc_path='GiveMeSomeCredit.csv',
            results_dir="./Results", figures_dir="./figures"),
        'quadratic': experiment_config(1, 'Give Me Some Credit', model='quadratic',
            seeds=[42, 43, 44, 45, 46], alpha=0.4, B=0.75, lambda_=30.0,
            tau=0.01, eta=0.2, T=500,
            cohort_size=256, fit_cap=5000, synthetic_n=6000,
            lambda_values=[3.0, 30.0],
            eta_values=[0.05, 0.2],
            baselines=dict(target_points=300, cost_quantiles=50),
            gmsc_path='GiveMeSomeCredit.csv',
            results_dir="./Results", figures_dir="./figures"),
    },

    'Synthetic nonlinear': {
        'affine': experiment_config(1, 'Synthetic nonlinear', model='affine',
            seeds=[42, 43, 44, 45, 46], alpha=0.4, B=0.75, lambda_=30.0,
            tau=0.01, eta=0.2, T=500,
            cohort_size=256, fit_cap=5000, synthetic_n=6000,
            lambda_values=[3.0, 30.0],
            eta_values=[0.05, 0.2],
            baselines=dict(target_points=300, cost_quantiles=50),
            results_dir="./Results", figures_dir="./figures"),
        'quadratic': experiment_config(1, 'Synthetic nonlinear', model='quadratic',
            seeds=[42, 43, 44, 45, 46], alpha=0.4, B=0.75, lambda_=30.0,
            tau=0.01, eta=0.2, T=500,
            cohort_size=256, fit_cap=5000, synthetic_n=6000,
            lambda_values=[3.0, 30.0],
            eta_values=[0.05, 0.2],
            baselines=dict(target_points=300, cost_quantiles=50),
            results_dir="./Results", figures_dir="./figures"),
    },

    'Synthetic curved': {
        'affine': experiment_config(1, 'Synthetic curved', model='affine',
            seeds=[42, 43, 44, 45, 46], alpha=0.4, B=0.75, lambda_=30.0,
            tau=0.01, eta=0.2, T=500,
            cohort_size=256, fit_cap=5000, synthetic_n=6000,
            lambda_values=[3.0, 30.0],
            eta_values=[0.05, 0.2],
            baselines=dict(target_points=300, cost_quantiles=50),
            results_dir="./Results", figures_dir="./figures"),
        'quadratic': experiment_config(1, 'Synthetic curved', model='quadratic',
            seeds=[42, 43, 44, 45, 46], alpha=0.4, B=0.75, lambda_=30.0,
            tau=0.01, eta=0.2, T=500,
            cohort_size=256, fit_cap=5000, synthetic_n=6000,
            lambda_values=[3.0, 30.0],
            eta_values=[0.05, 0.2],
            baselines=dict(target_points=300, cost_quantiles=50),
            results_dir="./Results", figures_dir="./figures"),
    },

}

# Selected pairs run sequentially; narrow the dataset list to run one dataset at a time.
experiment_1_selected_datasets = ['Synthetic curved', 'German Credit', 'Synthetic nonlinear', 'Give Me Some Credit', 'ACSIncome', 'Adult']
experiment_1_selected_models = ["affine", "quadratic"]

# Example local edits:
# experiment_1_selected_datasets = ["German Credit"]
# experiment_1_selected_models = ["affine"]
# experiment_1_configs["German Credit"]["affine"]["lambda_values"] = [3.0, 30.0]
# experiment_1_configs["German Credit"]["quadratic"]["retry_failed"] = True


In [ ]:
experiment_1_mode = "run"  # "run", "preflight", "load", or "skip"
experiment_1_selected_configs = select_experiment_configs(
    experiment_1_configs,
    experiment_1_selected_datasets,
    experiment_1_selected_models,
)
# Each lambda is a separate saved run and plot; matching fitted policies are cached.
experiment_1_run_configs = [
    dict(copy.deepcopy(config), lambda_=float(lambda_))
    for config in experiment_1_selected_configs
    for lambda_ in config["lambda_values"]
]
# A cheap plan: no data loading, fitting, or evaluation is triggered by this table.
display(pd.DataFrame([
    dict(dataset=config["dataset"], model=config["model"],
         lambda_values=(config["lambda_values"] if config["experiment"] == 2 else
                        [config["lambda_"]] if config["experiment"] != 7 else None),
         cases=len(experiment_cases(config)),
         policy_fit_requests=(0 if config["experiment"] == 7 else
             len(experiment_cases(config)) * sum(learned_method(m) is not None for m in config["methods"])))
    for config in experiment_1_run_configs
]))
experiment_1_reports = execute_experiment(
    experiment_1_run_configs,
    mode=experiment_1_mode,
)


## Experiment 2: Cost–validity trade-off

Use `lambda_values=[0.3, 1, 3, 10, 30]` at `B=0.75`, `alpha=0.4`, and `tau=0.01`. The low values provide cost-sensitive anchors; 3, 10, and 30 cover the regimes in which the relative performance of common and personalized targets changed. Keep every tested lambda in the results. Select checkpoints and baseline candidates on validation data; do not choose lambda from test outcomes.

All six datasets have independent affine/quadratic configurations below. Choose datasets and models with the two selection lists.


In [ ]:
# Independent configurations: edit any dataset/model pair without affecting the others.
# lambda_values is the executed lambda grid (experiment 7 does not sweep lambda).
# lambda_ is a scalar base value; execution below expands the listed lambda values.
experiment_2_configs = {
    'German Credit': {
        'affine': experiment_config(2, 'German Credit', model='affine',
            seeds=[45, 46], alpha=0.4, B=0.75, lambda_=30.0,
            tau=0.01, eta=0.2, T=500,
            cohort_size=256, fit_cap=5000, synthetic_n=6000,
            lambda_values=[0.3, 0.5, 1.0, 2.0, 3.0, 4.010.0, 20.0, 30.0, 50.0],
            baselines=dict(target_points=300, cost_quantiles=50),
            results_dir="./Results", figures_dir="./figures"),
        'quadratic': experiment_config(2, 'German Credit', model='quadratic',
            seeds=[45, 46], alpha=0.4, B=0.75, lambda_=30.0,
            tau=0.01, eta=0.2, T=500,
            cohort_size=512, fit_cap=5000, synthetic_n=6000,
            lambda_values=[0.3, 1.0, 2.0, 3.0, 10.0, 20.0, 30.0, 50.0],
            baselines=dict(target_points=300, cost_quantiles=50),
            results_dir="./Results", figures_dir="./figures"),
    },

    'Adult': {
        'affine': experiment_config(2, 'Adult', model='affine',
            seeds=[45, 46], alpha=0.4, B=0.75, lambda_=30.0,
            tau=0.01, eta=0.2, T=500,
            cohort_size=256, fit_cap=5000, synthetic_n=6000,
            lambda_values=[0.3, 1.0, 2.0, 3.0, 10.0, 20.0, 30.0, 50.0],
            baselines=dict(target_points=300, cost_quantiles=50),
            results_dir="./Results", figures_dir="./figures"),
        'quadratic': experiment_config(2, 'Adult', model='quadratic',
            seeds=[45, 46], alpha=0.4, B=0.75, lambda_=30.0,
            tau=0.01, eta=0.2, T=500,
            cohort_size=512, fit_cap=5000, synthetic_n=6000,
            lambda_values=[0.3, 1.0, 2.0, 3.0, 10.0, 20.0, 30.0, 50.0],
            baselines=dict(target_points=300, cost_quantiles=50),
            results_dir="./Results", figures_dir="./figures"),
    },

    'ACSIncome': {
        'affine': experiment_config(2, 'ACSIncome', model='affine',
            seeds=[45, 46], alpha=0.4, B=0.75, lambda_=30.0,
            tau=0.01, eta=0.2, T=500,
            cohort_size=256, fit_cap=5000, synthetic_n=6000,
            lambda_values=[0.3, 1.0, 2.0, 3.0, 10.0, 20.0, 30.0, 50.0],
            baselines=dict(target_points=300, cost_quantiles=50),
            results_dir="./Results", figures_dir="./figures"),
        'quadratic': experiment_config(2, 'ACSIncome', model='quadratic',
            seeds=[45, 46], alpha=0.4, B=0.75, lambda_=30.0,
            tau=0.01, eta=0.2, T=500,
            cohort_size=512, fit_cap=5000, synthetic_n=6000,
            lambda_values=[0.3, 1.0, 2.0, 3.0, 10.0, 20.0, 30.0, 50.0],
            baselines=dict(target_points=300, cost_quantiles=50),
            results_dir="./Results", figures_dir="./figures"),
    },

    'Give Me Some Credit': {
        'affine': experiment_config(2, 'Give Me Some Credit', model='affine',
            seeds=[45, 46], alpha=0.4, B=0.75, lambda_=30.0,
            tau=0.01, eta=0.2, T=500,
            cohort_size=256, fit_cap=5000, synthetic_n=6000,
            lambda_values=[0.3, 1.0, 2.0, 3.0, 10.0, 20.0, 30.0, 50.0],
            baselines=dict(target_points=300, cost_quantiles=50),
            gmsc_path='GiveMeSomeCredit.csv',
            results_dir="./Results", figures_dir="./figures"),
        'quadratic': experiment_config(2, 'Give Me Some Credit', model='quadratic',
            seeds=[45, 46], alpha=0.4, B=0.75, lambda_=30.0,
            tau=0.01, eta=0.2, T=500,
            cohort_size=256, fit_cap=5000, synthetic_n=6000,
            lambda_values=[0.3, 1.0, 2.0, 3.0, 10.0, 20.0, 30.0, 50.0],
            baselines=dict(target_points=300, cost_quantiles=50),
            gmsc_path='GiveMeSomeCredit.csv',
            results_dir="./Results", figures_dir="./figures"),
    },

    'Synthetic nonlinear': {
        'affine': experiment_config(2, 'Synthetic nonlinear', model='affine',
            seeds=[45, 46], alpha=0.4, B=0.75, lambda_=30.0,
            tau=0.01, eta=0.2, T=500,
            cohort_size=256, fit_cap=5000, synthetic_n=6000,
            lambda_values=[0.3, 1.0, 2.0, 3.0, 10.0, 20.0, 30.0, 50.0],
            baselines=dict(target_points=300, cost_quantiles=50),
            results_dir="./Results", figures_dir="./figures"),
        'quadratic': experiment_config(2, 'Synthetic nonlinear', model='quadratic',
            seeds=[45, 46], alpha=0.4, B=0.75, lambda_=30.0,
            tau=0.01, eta=0.2, T=500,
            cohort_size=512, fit_cap=5000, synthetic_n=6000,
            lambda_values=[0.3, 1.0, 2.0, 3.0, 10.0, 20.0, 30.0, 50.0],
            baselines=dict(target_points=300, cost_quantiles=50),
            results_dir="./Results", figures_dir="./figures"),
    },

    'Synthetic curved': {
        'affine': experiment_config(2, 'Synthetic curved', model='affine',
            seeds=[45, 46], alpha=0.4, B=0.75, lambda_=30.0,
            tau=0.01, eta=0.2, T=500,
            cohort_size=256, fit_cap=5000, synthetic_n=6000,
            lambda_values=[0.3, 1.0, 2.0, 3.0, 10.0, 20.0, 30.0, 50.0],
            baselines=dict(target_points=300, cost_quantiles=50),
            results_dir="./Results", figures_dir="./figures"),
        'quadratic': experiment_config(2, 'Synthetic curved', model='quadratic',
            seeds=[45, 46], alpha=0.4, B=0.75, lambda_=30.0,
            tau=0.01, eta=0.2, T=500,
            cohort_size=256, fit_cap=5000, synthetic_n=6000,
            lambda_values=[0.3, 1.0, 2.0, 3.0, 10.0, 20.0, 30.0, 50.0],
            baselines=dict(target_points=300, cost_quantiles=50),
            results_dir="./Results", figures_dir="./figures"),
    },

}

# Selected pairs run sequentially; narrow the dataset list to run one dataset at a time.
experiment_2_selected_datasets = ['Synthetic curved', 'German Credit', 'Synthetic nonlinear', 'Give Me Some Credit', 'ACSIncome', 'Adult']
experiment_2_selected_models = ["affine", "quadratic"]

# Example local edits:
# experiment_2_selected_datasets = ["German Credit"]
# experiment_2_selected_models = ["affine"]
# experiment_2_configs["German Credit"]["affine"]["lambda_values"] = [3.0, 30.0]
# experiment_2_configs["German Credit"]["quadratic"]["retry_failed"] = True


In [ ]:
experiment_2_mode = "run"  # "run", "preflight", "load", or "skip"
experiment_2_selected_configs = select_experiment_configs(
    experiment_2_configs,
    experiment_2_selected_datasets,
    experiment_2_selected_models,
)
experiment_2_run_configs = experiment_2_selected_configs
# A cheap plan: no data loading, fitting, or evaluation is triggered by this table.
display(pd.DataFrame([
    dict(dataset=config["dataset"], model=config["model"],
         lambda_values=(config["lambda_values"] if config["experiment"] == 2 else
                        [config["lambda_"]] if config["experiment"] != 7 else None),
         cases=len(experiment_cases(config)),
         policy_fit_requests=(0 if config["experiment"] == 7 else
             len(experiment_cases(config)) * sum(learned_method(m) is not None for m in config["methods"])))
    for config in experiment_2_run_configs
]))
experiment_2_reports = execute_experiment(
    experiment_2_run_configs,
    mode=experiment_2_mode,
)


## Experiment 3: Budget sensitivity

Run `B_values=[0.25, 0.75, 1.5]` separately for each value in `lambda_values=[3, 10, 30]`, fixing `alpha=0.4` and `tau=0.01`. Recompute eligibility and minimum-cost responses and retrain recourse at each budget, with the scorer and data split fixed within a seed. These levels cover limited reachability, the intermediate regime where personalization helped most, and the larger-budget regime where the common-target baseline became competitive. Each lambda has its own results and figures.

All six datasets have independent affine/quadratic configurations below. Choose datasets and models with the two selection lists.


In [ ]:
# Independent configurations: edit any dataset/model pair without affecting the others.
# lambda_values is the executed lambda grid (experiment 7 does not sweep lambda).
# lambda_ is a scalar base value; execution below expands the listed lambda values.
experiment_3_configs = {
    'German Credit': {
        'affine': experiment_config(3, 'German Credit', model='affine',
            seeds=[42, 43, 44, 45, 46], alpha=0.4, B=0.75, lambda_=30.0,
            tau=0.01, eta=0.2, T=500,
            cohort_size=256, fit_cap=5000, synthetic_n=6000,
            lambda_values=[3.0, 10.0, 30.0],
            B_values=[0.25, 0.75, 1.5],
            baselines=dict(target_points=300, cost_quantiles=50),
            results_dir="./Results", figures_dir="./figures"),
        'quadratic': experiment_config(3, 'German Credit', model='quadratic',
            seeds=[42, 43, 44, 45, 46], alpha=0.4, B=0.75, lambda_=30.0,
            tau=0.01, eta=0.2, T=500,
            cohort_size=256, fit_cap=5000, synthetic_n=6000,
            lambda_values=[3.0, 10.0, 30.0],
            B_values=[0.25, 0.75, 1.5],
            baselines=dict(target_points=300, cost_quantiles=50),
            results_dir="./Results", figures_dir="./figures"),
    },

    'Adult': {
        'affine': experiment_config(3, 'Adult', model='affine',
            seeds=[42, 43, 44, 45, 46], alpha=0.4, B=0.75, lambda_=30.0,
            tau=0.01, eta=0.2, T=500,
            cohort_size=256, fit_cap=5000, synthetic_n=6000,
            lambda_values=[3.0, 10.0, 30.0],
            B_values=[0.25, 0.75, 1.5],
            baselines=dict(target_points=300, cost_quantiles=50),
            results_dir="./Results", figures_dir="./figures"),
        'quadratic': experiment_config(3, 'Adult', model='quadratic',
            seeds=[42, 43, 44, 45, 46], alpha=0.4, B=0.75, lambda_=30.0,
            tau=0.01, eta=0.2, T=500,
            cohort_size=256, fit_cap=5000, synthetic_n=6000,
            lambda_values=[3.0, 10.0, 30.0],
            B_values=[0.25, 0.75, 1.5],
            baselines=dict(target_points=300, cost_quantiles=50),
            results_dir="./Results", figures_dir="./figures"),
    },

    'ACSIncome': {
        'affine': experiment_config(3, 'ACSIncome', model='affine',
            seeds=[42, 43, 44, 45, 46], alpha=0.4, B=0.75, lambda_=30.0,
            tau=0.01, eta=0.2, T=500,
            cohort_size=256, fit_cap=5000, synthetic_n=6000,
            lambda_values=[3.0, 10.0, 30.0],
            B_values=[0.25, 0.75, 1.5],
            baselines=dict(target_points=300, cost_quantiles=50),
            results_dir="./Results", figures_dir="./figures"),
        'quadratic': experiment_config(3, 'ACSIncome', model='quadratic',
            seeds=[42, 43, 44, 45, 46], alpha=0.4, B=0.75, lambda_=30.0,
            tau=0.01, eta=0.2, T=500,
            cohort_size=256, fit_cap=5000, synthetic_n=6000,
            lambda_values=[3.0, 10.0, 30.0],
            B_values=[0.25, 0.75, 1.5],
            baselines=dict(target_points=300, cost_quantiles=50),
            results_dir="./Results", figures_dir="./figures"),
    },

    'Give Me Some Credit': {
        'affine': experiment_config(3, 'Give Me Some Credit', model='affine',
            seeds=[42, 43, 44, 45, 46], alpha=0.4, B=0.75, lambda_=30.0,
            tau=0.01, eta=0.2, T=500,
            cohort_size=256, fit_cap=5000, synthetic_n=6000,
            lambda_values=[3.0, 10.0, 30.0],
            B_values=[0.25, 0.75, 1.5],
            baselines=dict(target_points=300, cost_quantiles=50),
            gmsc_path='GiveMeSomeCredit.csv',
            results_dir="./Results", figures_dir="./figures"),
        'quadratic': experiment_config(3, 'Give Me Some Credit', model='quadratic',
            seeds=[42, 43, 44, 45, 46], alpha=0.4, B=0.75, lambda_=30.0,
            tau=0.01, eta=0.2, T=500,
            cohort_size=256, fit_cap=5000, synthetic_n=6000,
            lambda_values=[3.0, 10.0, 30.0],
            B_values=[0.25, 0.75, 1.5],
            baselines=dict(target_points=300, cost_quantiles=50),
            gmsc_path='GiveMeSomeCredit.csv',
            results_dir="./Results", figures_dir="./figures"),
    },

    'Synthetic nonlinear': {
        'affine': experiment_config(3, 'Synthetic nonlinear', model='affine',
            seeds=[42, 43, 44, 45, 46], alpha=0.4, B=0.75, lambda_=30.0,
            tau=0.01, eta=0.2, T=500,
            cohort_size=256, fit_cap=5000, synthetic_n=6000,
            lambda_values=[3.0, 10.0, 30.0],
            B_values=[0.25, 0.75, 1.5],
            baselines=dict(target_points=300, cost_quantiles=50),
            results_dir="./Results", figures_dir="./figures"),
        'quadratic': experiment_config(3, 'Synthetic nonlinear', model='quadratic',
            seeds=[42, 43, 44, 45, 46], alpha=0.4, B=0.75, lambda_=30.0,
            tau=0.01, eta=0.2, T=500,
            cohort_size=256, fit_cap=5000, synthetic_n=6000,
            lambda_values=[3.0, 10.0, 30.0],
            B_values=[0.25, 0.75, 1.5],
            baselines=dict(target_points=300, cost_quantiles=50),
            results_dir="./Results", figures_dir="./figures"),
    },

    'Synthetic curved': {
        'affine': experiment_config(3, 'Synthetic curved', model='affine',
            seeds=[42, 43, 44, 45, 46], alpha=0.4, B=0.75, lambda_=30.0,
            tau=0.01, eta=0.2, T=500,
            cohort_size=256, fit_cap=5000, synthetic_n=6000,
            lambda_values=[3.0, 10.0, 30.0],
            B_values=[0.25, 0.75, 1.5],
            baselines=dict(target_points=300, cost_quantiles=50),
            results_dir="./Results", figures_dir="./figures"),
        'quadratic': experiment_config(3, 'Synthetic curved', model='quadratic',
            seeds=[42, 43, 44, 45, 46], alpha=0.4, B=0.75, lambda_=30.0,
            tau=0.01, eta=0.2, T=500,
            cohort_size=256, fit_cap=5000, synthetic_n=6000,
            lambda_values=[3.0, 10.0, 30.0],
            B_values=[0.25, 0.75, 1.5],
            baselines=dict(target_points=300, cost_quantiles=50),
            results_dir="./Results", figures_dir="./figures"),
    },

}

# Selected pairs run sequentially; narrow the dataset list to run one dataset at a time.
experiment_3_selected_datasets = ['Synthetic curved', 'German Credit', 'Synthetic nonlinear', 'Give Me Some Credit', 'ACSIncome', 'Adult']
experiment_3_selected_models = ["affine", "quadratic"]

# Example local edits:
# experiment_3_selected_datasets = ["German Credit"]
# experiment_3_selected_models = ["affine"]
# experiment_3_configs["German Credit"]["affine"]["lambda_values"] = [3.0, 30.0]
# experiment_3_configs["German Credit"]["quadratic"]["retry_failed"] = True


In [ ]:
experiment_3_mode = "run"  # "run", "preflight", "load", or "skip"
experiment_3_selected_configs = select_experiment_configs(
    experiment_3_configs,
    experiment_3_selected_datasets,
    experiment_3_selected_models,
)
# Each lambda is a separate saved run and plot; matching fitted policies are cached.
experiment_3_run_configs = [
    dict(copy.deepcopy(config), lambda_=float(lambda_))
    for config in experiment_3_selected_configs
    for lambda_ in config["lambda_values"]
]
# A cheap plan: no data loading, fitting, or evaluation is triggered by this table.
display(pd.DataFrame([
    dict(dataset=config["dataset"], model=config["model"],
         lambda_values=(config["lambda_values"] if config["experiment"] == 2 else
                        [config["lambda_"]] if config["experiment"] != 7 else None),
         cases=len(experiment_cases(config)),
         policy_fit_requests=(0 if config["experiment"] == 7 else
             len(experiment_cases(config)) * sum(learned_method(m) is not None for m in config["methods"])))
    for config in experiment_3_run_configs
]))
experiment_3_reports = execute_experiment(
    experiment_3_run_configs,
    mode=experiment_3_mode,
)


## Experiment 4: Admission-rate sensitivity

Run `alpha_values=[0.2, 0.4, 0.6]` separately for `lambda_values=[3, 10, 30]`, fixing `B=0.75` and `tau=0.01`. Recalibrate `t_0` on the same reference cohort and retrain recourse at each admission rate; reuse the scorer and split within a seed. The initially rejected group changes with alpha. Alongside conditional validity, the saved `V`, `N_minus`, and `k` permit reporting `V * N_minus / k`, the share of places going to that group. Each lambda has separate results and figures.

All six datasets have independent affine/quadratic configurations below. Choose datasets and models with the two selection lists.


In [ ]:
# Independent configurations: edit any dataset/model pair without affecting the others.
# lambda_values is the executed lambda grid (experiment 7 does not sweep lambda).
# lambda_ is a scalar base value; execution below expands the listed lambda values.
experiment_4_configs = {
    'German Credit': {
        'affine': experiment_config(4, 'German Credit', model='affine',
            seeds=[42, 43, 44, 45, 46], alpha=0.4, B=0.75, lambda_=30.0,
            tau=0.01, eta=0.2, T=500,
            cohort_size=256, fit_cap=5000, synthetic_n=6000,
            lambda_values=[3.0, 10.0, 30.0],
            alpha_values=[0.2, 0.4, 0.6],
            baselines=dict(target_points=300, cost_quantiles=50),
            results_dir="./Results", figures_dir="./figures"),
        'quadratic': experiment_config(4, 'German Credit', model='quadratic',
            seeds=[42, 43, 44, 45, 46], alpha=0.4, B=0.75, lambda_=30.0,
            tau=0.01, eta=0.2, T=500,
            cohort_size=256, fit_cap=5000, synthetic_n=6000,
            lambda_values=[3.0, 10.0, 30.0],
            alpha_values=[0.2, 0.4, 0.6],
            baselines=dict(target_points=300, cost_quantiles=50),
            results_dir="./Results", figures_dir="./figures"),
    },

    'Adult': {
        'affine': experiment_config(4, 'Adult', model='affine',
            seeds=[42, 43, 44, 45, 46], alpha=0.4, B=0.75, lambda_=30.0,
            tau=0.01, eta=0.2, T=500,
            cohort_size=256, fit_cap=5000, synthetic_n=6000,
            lambda_values=[3.0, 10.0, 30.0],
            alpha_values=[0.2, 0.4, 0.6],
            baselines=dict(target_points=300, cost_quantiles=50),
            results_dir="./Results", figures_dir="./figures"),
        'quadratic': experiment_config(4, 'Adult', model='quadratic',
            seeds=[42, 43, 44, 45, 46], alpha=0.4, B=0.75, lambda_=30.0,
            tau=0.01, eta=0.2, T=500,
            cohort_size=256, fit_cap=5000, synthetic_n=6000,
            lambda_values=[3.0, 10.0, 30.0],
            alpha_values=[0.2, 0.4, 0.6],
            baselines=dict(target_points=300, cost_quantiles=50),
            results_dir="./Results", figures_dir="./figures"),
    },

    'ACSIncome': {
        'affine': experiment_config(4, 'ACSIncome', model='affine',
            seeds=[42, 43, 44, 45, 46], alpha=0.4, B=0.75, lambda_=30.0,
            tau=0.01, eta=0.2, T=500,
            cohort_size=256, fit_cap=5000, synthetic_n=6000,
            lambda_values=[3.0, 10.0, 30.0],
            alpha_values=[0.2, 0.4, 0.6],
            baselines=dict(target_points=300, cost_quantiles=50),
            results_dir="./Results", figures_dir="./figures"),
        'quadratic': experiment_config(4, 'ACSIncome', model='quadratic',
            seeds=[42, 43, 44, 45, 46], alpha=0.4, B=0.75, lambda_=30.0,
            tau=0.01, eta=0.2, T=500,
            cohort_size=256, fit_cap=5000, synthetic_n=6000,
            lambda_values=[3.0, 10.0, 30.0],
            alpha_values=[0.2, 0.4, 0.6],
            baselines=dict(target_points=300, cost_quantiles=50),
            results_dir="./Results", figures_dir="./figures"),
    },

    'Give Me Some Credit': {
        'affine': experiment_config(4, 'Give Me Some Credit', model='affine',
            seeds=[42, 43, 44, 45, 46], alpha=0.4, B=0.75, lambda_=30.0,
            tau=0.01, eta=0.2, T=500,
            cohort_size=256, fit_cap=5000, synthetic_n=6000,
            lambda_values=[3.0, 10.0, 30.0],
            alpha_values=[0.2, 0.4, 0.6],
            baselines=dict(target_points=300, cost_quantiles=50),
            gmsc_path='GiveMeSomeCredit.csv',
            results_dir="./Results", figures_dir="./figures"),
        'quadratic': experiment_config(4, 'Give Me Some Credit', model='quadratic',
            seeds=[42, 43, 44, 45, 46], alpha=0.4, B=0.75, lambda_=30.0,
            tau=0.01, eta=0.2, T=500,
            cohort_size=256, fit_cap=5000, synthetic_n=6000,
            lambda_values=[3.0, 10.0, 30.0],
            alpha_values=[0.2, 0.4, 0.6],
            baselines=dict(target_points=300, cost_quantiles=50),
            gmsc_path='GiveMeSomeCredit.csv',
            results_dir="./Results", figures_dir="./figures"),
    },

    'Synthetic nonlinear': {
        'affine': experiment_config(4, 'Synthetic nonlinear', model='affine',
            seeds=[42, 43, 44, 45, 46], alpha=0.4, B=0.75, lambda_=30.0,
            tau=0.01, eta=0.2, T=500,
            cohort_size=256, fit_cap=5000, synthetic_n=6000,
            lambda_values=[3.0, 10.0, 30.0],
            alpha_values=[0.2, 0.4, 0.6],
            baselines=dict(target_points=300, cost_quantiles=50),
            results_dir="./Results", figures_dir="./figures"),
        'quadratic': experiment_config(4, 'Synthetic nonlinear', model='quadratic',
            seeds=[42, 43, 44, 45, 46], alpha=0.4, B=0.75, lambda_=30.0,
            tau=0.01, eta=0.2, T=500,
            cohort_size=256, fit_cap=5000, synthetic_n=6000,
            lambda_values=[3.0, 10.0, 30.0],
            alpha_values=[0.2, 0.4, 0.6],
            baselines=dict(target_points=300, cost_quantiles=50),
            results_dir="./Results", figures_dir="./figures"),
    },

    'Synthetic curved': {
        'affine': experiment_config(4, 'Synthetic curved', model='affine',
            seeds=[42, 43, 44, 45, 46], alpha=0.4, B=0.75, lambda_=30.0,
            tau=0.01, eta=0.2, T=500,
            cohort_size=256, fit_cap=5000, synthetic_n=6000,
            lambda_values=[3.0, 10.0, 30.0],
            alpha_values=[0.2, 0.4, 0.6],
            baselines=dict(target_points=300, cost_quantiles=50),
            results_dir="./Results", figures_dir="./figures"),
        'quadratic': experiment_config(4, 'Synthetic curved', model='quadratic',
            seeds=[42, 43, 44, 45, 46], alpha=0.4, B=0.75, lambda_=30.0,
            tau=0.01, eta=0.2, T=500,
            cohort_size=256, fit_cap=5000, synthetic_n=6000,
            lambda_values=[3.0, 10.0, 30.0],
            alpha_values=[0.2, 0.4, 0.6],
            baselines=dict(target_points=300, cost_quantiles=50),
            results_dir="./Results", figures_dir="./figures"),
    },

}

# Selected pairs run sequentially; narrow the dataset list to run one dataset at a time.
experiment_4_selected_datasets = ['Synthetic curved', 'German Credit', 'Synthetic nonlinear', 'Give Me Some Credit', 'ACSIncome', 'Adult']
experiment_4_selected_models = ["affine", "quadratic"]

# Example local edits:
# experiment_4_selected_datasets = ["German Credit"]
# experiment_4_selected_models = ["affine"]
# experiment_4_configs["German Credit"]["affine"]["lambda_values"] = [3.0, 30.0]
# experiment_4_configs["German Credit"]["quadratic"]["retry_failed"] = True


In [ ]:
experiment_4_mode = "run"  # "run", "preflight", "load", or "skip"
experiment_4_selected_configs = select_experiment_configs(
    experiment_4_configs,
    experiment_4_selected_datasets,
    experiment_4_selected_models,
)
# Each lambda is a separate saved run and plot; matching fitted policies are cached.
experiment_4_run_configs = [
    dict(copy.deepcopy(config), lambda_=float(lambda_))
    for config in experiment_4_selected_configs
    for lambda_ in config["lambda_values"]
]
# A cheap plan: no data loading, fitting, or evaluation is triggered by this table.
display(pd.DataFrame([
    dict(dataset=config["dataset"], model=config["model"],
         lambda_values=(config["lambda_values"] if config["experiment"] == 2 else
                        [config["lambda_"]] if config["experiment"] != 7 else None),
         cases=len(experiment_cases(config)),
         policy_fit_requests=(0 if config["experiment"] == 7 else
             len(experiment_cases(config)) * sum(learned_method(m) is not None for m in config["methods"])))
    for config in experiment_4_run_configs
]))
experiment_4_reports = execute_experiment(
    experiment_4_run_configs,
    mode=experiment_4_mode,
)


## Experiment 5: Smoothing-temperature sensitivity

Use `tau_values=[0.005, 0.01, 0.02, 0.05]` at `lambda_values=[30]`, `B=0.75`, and `alpha=0.4`. Hold `eta=0.05` fixed across temperatures so the comparison changes only smoothing. This grid brackets the stable `tau=0.01` setting; the smaller step size limits numerical risk at the sharper endpoint. Save the smoothing-versus-exact-allocation discrepancy for the same fractional recommendation mixture. Additional lambda values can be added locally, but the default avoids a large cross-product.

All six datasets have independent affine/quadratic configurations below. Choose datasets and models with the two selection lists.


In [ ]:
# Independent configurations: edit any dataset/model pair without affecting the others.
# lambda_values is the executed lambda grid (experiment 7 does not sweep lambda).
# lambda_ is a scalar base value; execution below expands the listed lambda values.
experiment_5_configs = {
    'German Credit': {
        'affine': experiment_config(5, 'German Credit', model='affine',
            seeds=[42, 43, 44, 45, 46], alpha=0.4, B=0.75, lambda_=30.0,
            tau=0.01, eta=0.05, T=500,
            cohort_size=256, fit_cap=5000, synthetic_n=6000,
            lambda_values=[30.0],
            tau_values=[0.005, 0.01, 0.02, 0.05],
            baselines=dict(target_points=300, cost_quantiles=50),
            results_dir="./Results", figures_dir="./figures"),
        'quadratic': experiment_config(5, 'German Credit', model='quadratic',
            seeds=[42, 43, 44, 45, 46], alpha=0.4, B=0.75, lambda_=30.0,
            tau=0.01, eta=0.05, T=500,
            cohort_size=256, fit_cap=5000, synthetic_n=6000,
            lambda_values=[30.0],
            tau_values=[0.005, 0.01, 0.02, 0.05],
            baselines=dict(target_points=300, cost_quantiles=50),
            results_dir="./Results", figures_dir="./figures"),
    },

    'Adult': {
        'affine': experiment_config(5, 'Adult', model='affine',
            seeds=[42, 43, 44, 45, 46], alpha=0.4, B=0.75, lambda_=30.0,
            tau=0.01, eta=0.05, T=500,
            cohort_size=256, fit_cap=5000, synthetic_n=6000,
            lambda_values=[30.0],
            tau_values=[0.005, 0.01, 0.02, 0.05],
            baselines=dict(target_points=300, cost_quantiles=50),
            results_dir="./Results", figures_dir="./figures"),
        'quadratic': experiment_config(5, 'Adult', model='quadratic',
            seeds=[42, 43, 44, 45, 46], alpha=0.4, B=0.75, lambda_=30.0,
            tau=0.01, eta=0.05, T=500,
            cohort_size=256, fit_cap=5000, synthetic_n=6000,
            lambda_values=[30.0],
            tau_values=[0.005, 0.01, 0.02, 0.05],
            baselines=dict(target_points=300, cost_quantiles=50),
            results_dir="./Results", figures_dir="./figures"),
    },

    'ACSIncome': {
        'affine': experiment_config(5, 'ACSIncome', model='affine',
            seeds=[42, 43, 44, 45, 46], alpha=0.4, B=0.75, lambda_=30.0,
            tau=0.01, eta=0.05, T=500,
            cohort_size=256, fit_cap=5000, synthetic_n=6000,
            lambda_values=[30.0],
            tau_values=[0.005, 0.01, 0.02, 0.05],
            baselines=dict(target_points=300, cost_quantiles=50),
            results_dir="./Results", figures_dir="./figures"),
        'quadratic': experiment_config(5, 'ACSIncome', model='quadratic',
            seeds=[42, 43, 44, 45, 46], alpha=0.4, B=0.75, lambda_=30.0,
            tau=0.01, eta=0.05, T=500,
            cohort_size=256, fit_cap=5000, synthetic_n=6000,
            lambda_values=[30.0],
            tau_values=[0.005, 0.01, 0.02, 0.05],
            baselines=dict(target_points=300, cost_quantiles=50),
            results_dir="./Results", figures_dir="./figures"),
    },

    'Give Me Some Credit': {
        'affine': experiment_config(5, 'Give Me Some Credit', model='affine',
            seeds=[42, 43, 44, 45, 46], alpha=0.4, B=0.75, lambda_=30.0,
            tau=0.01, eta=0.05, T=500,
            cohort_size=256, fit_cap=5000, synthetic_n=6000,
            lambda_values=[30.0],
            tau_values=[0.005, 0.01, 0.02, 0.05],
            baselines=dict(target_points=300, cost_quantiles=50),
            gmsc_path='GiveMeSomeCredit.csv',
            results_dir="./Results", figures_dir="./figures"),
        'quadratic': experiment_config(5, 'Give Me Some Credit', model='quadratic',
            seeds=[42, 43, 44, 45, 46], alpha=0.4, B=0.75, lambda_=30.0,
            tau=0.01, eta=0.05, T=500,
            cohort_size=256, fit_cap=5000, synthetic_n=6000,
            lambda_values=[30.0],
            tau_values=[0.005, 0.01, 0.02, 0.05],
            baselines=dict(target_points=300, cost_quantiles=50),
            gmsc_path='GiveMeSomeCredit.csv',
            results_dir="./Results", figures_dir="./figures"),
    },

    'Synthetic nonlinear': {
        'affine': experiment_config(5, 'Synthetic nonlinear', model='affine',
            seeds=[42, 43, 44, 45, 46], alpha=0.4, B=0.75, lambda_=30.0,
            tau=0.01, eta=0.05, T=500,
            cohort_size=256, fit_cap=5000, synthetic_n=6000,
            lambda_values=[30.0],
            tau_values=[0.005, 0.01, 0.02, 0.05],
            baselines=dict(target_points=300, cost_quantiles=50),
            results_dir="./Results", figures_dir="./figures"),
        'quadratic': experiment_config(5, 'Synthetic nonlinear', model='quadratic',
            seeds=[42, 43, 44, 45, 46], alpha=0.4, B=0.75, lambda_=30.0,
            tau=0.01, eta=0.05, T=500,
            cohort_size=256, fit_cap=5000, synthetic_n=6000,
            lambda_values=[30.0],
            tau_values=[0.005, 0.01, 0.02, 0.05],
            baselines=dict(target_points=300, cost_quantiles=50),
            results_dir="./Results", figures_dir="./figures"),
    },

    'Synthetic curved': {
        'affine': experiment_config(5, 'Synthetic curved', model='affine',
            seeds=[42, 43, 44, 45, 46], alpha=0.4, B=0.75, lambda_=30.0,
            tau=0.01, eta=0.05, T=500,
            cohort_size=256, fit_cap=5000, synthetic_n=6000,
            lambda_values=[30.0],
            tau_values=[0.005, 0.01, 0.02, 0.05],
            baselines=dict(target_points=300, cost_quantiles=50),
            results_dir="./Results", figures_dir="./figures"),
        'quadratic': experiment_config(5, 'Synthetic curved', model='quadratic',
            seeds=[42, 43, 44, 45, 46], alpha=0.4, B=0.75, lambda_=30.0,
            tau=0.01, eta=0.05, T=500,
            cohort_size=256, fit_cap=5000, synthetic_n=6000,
            lambda_values=[30.0],
            tau_values=[0.005, 0.01, 0.02, 0.05],
            baselines=dict(target_points=300, cost_quantiles=50),
            results_dir="./Results", figures_dir="./figures"),
    },

}

# Selected pairs run sequentially; narrow the dataset list to run one dataset at a time.
experiment_5_selected_datasets = ['Synthetic curved', 'German Credit', 'Synthetic nonlinear', 'Give Me Some Credit', 'ACSIncome', 'Adult']
experiment_5_selected_models = ["affine", "quadratic"]

# Example local edits:
# experiment_5_selected_datasets = ["German Credit"]
# experiment_5_selected_models = ["affine"]
# experiment_5_configs["German Credit"]["affine"]["lambda_values"] = [3.0, 30.0]
# experiment_5_configs["German Credit"]["quadratic"]["retry_failed"] = True


In [ ]:
experiment_5_mode = "run"  # "run", "preflight", "load", or "skip"
experiment_5_selected_configs = select_experiment_configs(
    experiment_5_configs,
    experiment_5_selected_datasets,
    experiment_5_selected_models,
)
# Each lambda is a separate saved run and plot; matching fitted policies are cached.
experiment_5_run_configs = [
    dict(copy.deepcopy(config), lambda_=float(lambda_))
    for config in experiment_5_selected_configs
    for lambda_ in config["lambda_values"]
]
# A cheap plan: no data loading, fitting, or evaluation is triggered by this table.
display(pd.DataFrame([
    dict(dataset=config["dataset"], model=config["model"],
         lambda_values=(config["lambda_values"] if config["experiment"] == 2 else
                        [config["lambda_"]] if config["experiment"] != 7 else None),
         cases=len(experiment_cases(config)),
         policy_fit_requests=(0 if config["experiment"] == 7 else
             len(experiment_cases(config)) * sum(learned_method(m) is not None for m in config["methods"])))
    for config in experiment_5_run_configs
]))
experiment_5_reports = execute_experiment(
    experiment_5_run_configs,
    mode=experiment_5_mode,
)


## Experiment 6: Baselines and ablations

Compare all configured baselines and ablations separately at `lambda_values=[3, 10, 30]`, with `B=0.75`, `alpha=0.4`, and `tau=0.01`. Retain no recourse, original-threshold recourse, and the common target with cost selection: the latter was strong when costs received more weight. Use 300 target-grid points and 50 cost quantiles. The fixed-threshold and detached-threshold ablations assess the role of competition; the full method also saves its gradient decomposition. Each lambda has separate figures and result files.

All six datasets have independent affine/quadratic configurations below. Choose datasets and models with the two selection lists.


In [ ]:
# Independent configurations: edit any dataset/model pair without affecting the others.
# lambda_values is the executed lambda grid (experiment 7 does not sweep lambda).
# lambda_ is a scalar base value; execution below expands the listed lambda values.
experiment_6_configs = {
    'German Credit': {
        'affine': experiment_config(6, 'German Credit', model='affine',
            seeds=[42, 43, 44, 45, 46], alpha=0.4, B=0.75, lambda_=30.0,
            tau=0.01, eta=0.2, T=500,
            cohort_size=256, fit_cap=5000, synthetic_n=6000,
            lambda_values=[3.0, 10.0, 30.0],
            methods=['No recourse', 'Original threshold', 'Common target', 'Common target + cost selection', 'Iterative common target', 'Personalized targets + selection', 'Personalized targets, all eligible', 'Fixed t_0 (ablation)', 'Detached threshold (ablation)'],
            baselines=dict(target_points=300, cost_quantiles=50),
            results_dir="./Results", figures_dir="./figures"),
        'quadratic': experiment_config(6, 'German Credit', model='quadratic',
            seeds=[42, 43, 44, 45, 46], alpha=0.4, B=0.75, lambda_=30.0,
            tau=0.01, eta=0.2, T=500,
            cohort_size=256, fit_cap=5000, synthetic_n=6000,
            lambda_values=[3.0, 10.0, 30.0],
            methods=['No recourse', 'Original threshold', 'Common target', 'Common target + cost selection', 'Iterative common target', 'Personalized targets + selection', 'Personalized targets, all eligible', 'Fixed t_0 (ablation)', 'Detached threshold (ablation)'],
            baselines=dict(target_points=300, cost_quantiles=50),
            results_dir="./Results", figures_dir="./figures"),
    },

    'Adult': {
        'affine': experiment_config(6, 'Adult', model='affine',
            seeds=[42, 43, 44, 45, 46], alpha=0.4, B=0.75, lambda_=30.0,
            tau=0.01, eta=0.2, T=500,
            cohort_size=256, fit_cap=5000, synthetic_n=6000,
            lambda_values=[3.0, 10.0, 30.0],
            methods=['No recourse', 'Original threshold', 'Common target', 'Common target + cost selection', 'Iterative common target', 'Personalized targets + selection', 'Personalized targets, all eligible', 'Fixed t_0 (ablation)', 'Detached threshold (ablation)'],
            baselines=dict(target_points=300, cost_quantiles=50),
            results_dir="./Results", figures_dir="./figures"),
        'quadratic': experiment_config(6, 'Adult', model='quadratic',
            seeds=[42, 43, 44, 45, 46], alpha=0.4, B=0.75, lambda_=30.0,
            tau=0.01, eta=0.2, T=500,
            cohort_size=256, fit_cap=5000, synthetic_n=6000,
            lambda_values=[3.0, 10.0, 30.0],
            methods=['No recourse', 'Original threshold', 'Common target', 'Common target + cost selection', 'Iterative common target', 'Personalized targets + selection', 'Personalized targets, all eligible', 'Fixed t_0 (ablation)', 'Detached threshold (ablation)'],
            baselines=dict(target_points=300, cost_quantiles=50),
            results_dir="./Results", figures_dir="./figures"),
    },

    'ACSIncome': {
        'affine': experiment_config(6, 'ACSIncome', model='affine',
            seeds=[42, 43, 44, 45, 46], alpha=0.4, B=0.75, lambda_=30.0,
            tau=0.01, eta=0.2, T=500,
            cohort_size=256, fit_cap=5000, synthetic_n=6000,
            lambda_values=[3.0, 10.0, 30.0],
            methods=['No recourse', 'Original threshold', 'Common target', 'Common target + cost selection', 'Iterative common target', 'Personalized targets + selection', 'Personalized targets, all eligible', 'Fixed t_0 (ablation)', 'Detached threshold (ablation)'],
            baselines=dict(target_points=300, cost_quantiles=50),
            results_dir="./Results", figures_dir="./figures"),
        'quadratic': experiment_config(6, 'ACSIncome', model='quadratic',
            seeds=[42, 43, 44, 45, 46], alpha=0.4, B=0.75, lambda_=30.0,
            tau=0.01, eta=0.2, T=500,
            cohort_size=256, fit_cap=5000, synthetic_n=6000,
            lambda_values=[3.0, 10.0, 30.0],
            methods=['No recourse', 'Original threshold', 'Common target', 'Common target + cost selection', 'Iterative common target', 'Personalized targets + selection', 'Personalized targets, all eligible', 'Fixed t_0 (ablation)', 'Detached threshold (ablation)'],
            baselines=dict(target_points=300, cost_quantiles=50),
            results_dir="./Results", figures_dir="./figures"),
    },

    'Give Me Some Credit': {
        'affine': experiment_config(6, 'Give Me Some Credit', model='affine',
            seeds=[42, 43, 44, 45, 46], alpha=0.4, B=0.75, lambda_=30.0,
            tau=0.01, eta=0.2, T=500,
            cohort_size=256, fit_cap=5000, synthetic_n=6000,
            lambda_values=[3.0, 10.0, 30.0],
            methods=['No recourse', 'Original threshold', 'Common target', 'Common target + cost selection', 'Iterative common target', 'Personalized targets + selection', 'Personalized targets, all eligible', 'Fixed t_0 (ablation)', 'Detached threshold (ablation)'],
            baselines=dict(target_points=300, cost_quantiles=50),
            gmsc_path='GiveMeSomeCredit.csv',
            results_dir="./Results", figures_dir="./figures"),
        'quadratic': experiment_config(6, 'Give Me Some Credit', model='quadratic',
            seeds=[42, 43, 44, 45, 46], alpha=0.4, B=0.75, lambda_=30.0,
            tau=0.01, eta=0.2, T=500,
            cohort_size=256, fit_cap=5000, synthetic_n=6000,
            lambda_values=[3.0, 10.0, 30.0],
            methods=['No recourse', 'Original threshold', 'Common target', 'Common target + cost selection', 'Iterative common target', 'Personalized targets + selection', 'Personalized targets, all eligible', 'Fixed t_0 (ablation)', 'Detached threshold (ablation)'],
            baselines=dict(target_points=300, cost_quantiles=50),
            gmsc_path='GiveMeSomeCredit.csv',
            results_dir="./Results", figures_dir="./figures"),
    },

    'Synthetic nonlinear': {
        'affine': experiment_config(6, 'Synthetic nonlinear', model='affine',
            seeds=[42, 43, 44, 45, 46], alpha=0.4, B=0.75, lambda_=30.0,
            tau=0.01, eta=0.2, T=500,
            cohort_size=256, fit_cap=5000, synthetic_n=6000,
            lambda_values=[3.0, 10.0, 30.0],
            methods=['No recourse', 'Original threshold', 'Common target', 'Common target + cost selection', 'Iterative common target', 'Personalized targets + selection', 'Personalized targets, all eligible', 'Fixed t_0 (ablation)', 'Detached threshold (ablation)'],
            baselines=dict(target_points=300, cost_quantiles=50),
            results_dir="./Results", figures_dir="./figures"),
        'quadratic': experiment_config(6, 'Synthetic nonlinear', model='quadratic',
            seeds=[42, 43, 44, 45, 46], alpha=0.4, B=0.75, lambda_=30.0,
            tau=0.01, eta=0.2, T=500,
            cohort_size=256, fit_cap=5000, synthetic_n=6000,
            lambda_values=[3.0, 10.0, 30.0],
            methods=['No recourse', 'Original threshold', 'Common target', 'Common target + cost selection', 'Iterative common target', 'Personalized targets + selection', 'Personalized targets, all eligible', 'Fixed t_0 (ablation)', 'Detached threshold (ablation)'],
            baselines=dict(target_points=300, cost_quantiles=50),
            results_dir="./Results", figures_dir="./figures"),
    },

    'Synthetic curved': {
        'affine': experiment_config(6, 'Synthetic curved', model='affine',
            seeds=[42, 43, 44, 45, 46], alpha=0.4, B=0.75, lambda_=30.0,
            tau=0.01, eta=0.2, T=500,
            cohort_size=256, fit_cap=5000, synthetic_n=6000,
            lambda_values=[3.0, 10.0, 30.0],
            methods=['No recourse', 'Original threshold', 'Common target', 'Common target + cost selection', 'Iterative common target', 'Personalized targets + selection', 'Personalized targets, all eligible', 'Fixed t_0 (ablation)', 'Detached threshold (ablation)'],
            baselines=dict(target_points=300, cost_quantiles=50),
            results_dir="./Results", figures_dir="./figures"),
        'quadratic': experiment_config(6, 'Synthetic curved', model='quadratic',
            seeds=[42, 43, 44, 45, 46], alpha=0.4, B=0.75, lambda_=30.0,
            tau=0.01, eta=0.2, T=500,
            cohort_size=256, fit_cap=5000, synthetic_n=6000,
            lambda_values=[3.0, 10.0, 30.0],
            methods=['No recourse', 'Original threshold', 'Common target', 'Common target + cost selection', 'Iterative common target', 'Personalized targets + selection', 'Personalized targets, all eligible', 'Fixed t_0 (ablation)', 'Detached threshold (ablation)'],
            baselines=dict(target_points=300, cost_quantiles=50),
            results_dir="./Results", figures_dir="./figures"),
    },

}

# Selected pairs run sequentially; narrow the dataset list to run one dataset at a time.
experiment_6_selected_datasets = ['Synthetic curved', 'German Credit', 'Synthetic nonlinear', 'Give Me Some Credit', 'ACSIncome', 'Adult']
experiment_6_selected_models = ["affine", "quadratic"]

# Example local edits:
# experiment_6_selected_datasets = ["German Credit"]
# experiment_6_selected_models = ["affine"]
# experiment_6_configs["German Credit"]["affine"]["lambda_values"] = [3.0, 30.0]
# experiment_6_configs["German Credit"]["quadratic"]["retry_failed"] = True


In [ ]:
experiment_6_mode = "run"  # "run", "preflight", "load", or "skip"
experiment_6_selected_configs = select_experiment_configs(
    experiment_6_configs,
    experiment_6_selected_datasets,
    experiment_6_selected_models,
)
# Each lambda is a separate saved run and plot; matching fitted policies are cached.
experiment_6_run_configs = [
    dict(copy.deepcopy(config), lambda_=float(lambda_))
    for config in experiment_6_selected_configs
    for lambda_ in config["lambda_values"]
]
# A cheap plan: no data loading, fitting, or evaluation is triggered by this table.
display(pd.DataFrame([
    dict(dataset=config["dataset"], model=config["model"],
         lambda_values=(config["lambda_values"] if config["experiment"] == 2 else
                        [config["lambda_"]] if config["experiment"] != 7 else None),
         cases=len(experiment_cases(config)),
         policy_fit_requests=(0 if config["experiment"] == 7 else
             len(experiment_cases(config)) * sum(learned_method(m) is not None for m in config["methods"])))
    for config in experiment_6_run_configs
]))
experiment_6_reports = execute_experiment(
    experiment_6_run_configs,
    mode=experiment_6_mode,
)


## Experiment 7: Solver and predictive-model audit

Audit both affine and quadratic response solvers at reachable-interval fractions `[0.05, 0.25, 0.5, 0.75, 0.95]` for up to 32 eligible applicants per seed. The added endpoints probe targets near the original threshold and the reachable-score limit. Use `B=0.75` and `alpha=0.4`; no policy training or lambda sweep is needed. Save score-attainment, immutable-coordinate, budget, and solver diagnostics, plus predictive test metrics when labels are available.

All six datasets have independent affine/quadratic configurations below. Choose datasets and models with the two selection lists.


In [ ]:
# Independent configurations: edit any dataset/model pair without affecting the others.
# lambda_values is the executed lambda grid (experiment 7 does not sweep lambda).
# lambda_ is a scalar base value; execution below expands the listed lambda values.
experiment_7_configs = {
    'German Credit': {
        'affine': experiment_config(7, 'German Credit', model='affine',
            seeds=[42, 43, 44, 45, 46], alpha=0.4, B=0.75, lambda_=30.0,
            tau=0.01, eta=0.2, T=500,
            cohort_size=256, fit_cap=5000, synthetic_n=6000,
            audit_fractions=[0.05, 0.25, 0.5, 0.75, 0.95],
            audit_limit=32,
            baselines=dict(target_points=300, cost_quantiles=50),
            results_dir="./Results", figures_dir="./figures"),
        'quadratic': experiment_config(7, 'German Credit', model='quadratic',
            seeds=[42, 43, 44, 45, 46], alpha=0.4, B=0.75, lambda_=30.0,
            tau=0.01, eta=0.2, T=500,
            cohort_size=256, fit_cap=5000, synthetic_n=6000,
            audit_fractions=[0.05, 0.25, 0.5, 0.75, 0.95],
            audit_limit=32,
            baselines=dict(target_points=300, cost_quantiles=50),
            results_dir="./Results", figures_dir="./figures"),
    },

    'Adult': {
        'affine': experiment_config(7, 'Adult', model='affine',
            seeds=[42, 43, 44, 45, 46], alpha=0.4, B=0.75, lambda_=30.0,
            tau=0.01, eta=0.2, T=500,
            cohort_size=256, fit_cap=5000, synthetic_n=6000,
            audit_fractions=[0.05, 0.25, 0.5, 0.75, 0.95],
            audit_limit=32,
            baselines=dict(target_points=300, cost_quantiles=50),
            results_dir="./Results", figures_dir="./figures"),
        'quadratic': experiment_config(7, 'Adult', model='quadratic',
            seeds=[42, 43, 44, 45, 46], alpha=0.4, B=0.75, lambda_=30.0,
            tau=0.01, eta=0.2, T=500,
            cohort_size=256, fit_cap=5000, synthetic_n=6000,
            audit_fractions=[0.05, 0.25, 0.5, 0.75, 0.95],
            audit_limit=32,
            baselines=dict(target_points=300, cost_quantiles=50),
            results_dir="./Results", figures_dir="./figures"),
    },

    'ACSIncome': {
        'affine': experiment_config(7, 'ACSIncome', model='affine',
            seeds=[42, 43, 44, 45, 46], alpha=0.4, B=0.75, lambda_=30.0,
            tau=0.01, eta=0.2, T=500,
            cohort_size=256, fit_cap=5000, synthetic_n=6000,
            audit_fractions=[0.05, 0.25, 0.5, 0.75, 0.95],
            audit_limit=32,
            baselines=dict(target_points=300, cost_quantiles=50),
            results_dir="./Results", figures_dir="./figures"),
        'quadratic': experiment_config(7, 'ACSIncome', model='quadratic',
            seeds=[42, 43, 44, 45, 46], alpha=0.4, B=0.75, lambda_=30.0,
            tau=0.01, eta=0.2, T=500,
            cohort_size=256, fit_cap=5000, synthetic_n=6000,
            audit_fractions=[0.05, 0.25, 0.5, 0.75, 0.95],
            audit_limit=32,
            baselines=dict(target_points=300, cost_quantiles=50),
            results_dir="./Results", figures_dir="./figures"),
    },

    'Give Me Some Credit': {
        'affine': experiment_config(7, 'Give Me Some Credit', model='affine',
            seeds=[42, 43, 44, 45, 46], alpha=0.4, B=0.75, lambda_=30.0,
            tau=0.01, eta=0.2, T=500,
            cohort_size=256, fit_cap=5000, synthetic_n=6000,
            audit_fractions=[0.05, 0.25, 0.5, 0.75, 0.95],
            audit_limit=32,
            baselines=dict(target_points=300, cost_quantiles=50),
            gmsc_path='GiveMeSomeCredit.csv',
            results_dir="./Results", figures_dir="./figures"),
        'quadratic': experiment_config(7, 'Give Me Some Credit', model='quadratic',
            seeds=[42, 43, 44, 45, 46], alpha=0.4, B=0.75, lambda_=30.0,
            tau=0.01, eta=0.2, T=500,
            cohort_size=256, fit_cap=5000, synthetic_n=6000,
            audit_fractions=[0.05, 0.25, 0.5, 0.75, 0.95],
            audit_limit=32,
            baselines=dict(target_points=300, cost_quantiles=50),
            gmsc_path='GiveMeSomeCredit.csv',
            results_dir="./Results", figures_dir="./figures"),
    },

    'Synthetic nonlinear': {
        'affine': experiment_config(7, 'Synthetic nonlinear', model='affine',
            seeds=[42, 43, 44, 45, 46], alpha=0.4, B=0.75, lambda_=30.0,
            tau=0.01, eta=0.2, T=500,
            cohort_size=256, fit_cap=5000, synthetic_n=6000,
            audit_fractions=[0.05, 0.25, 0.5, 0.75, 0.95],
            audit_limit=32,
            baselines=dict(target_points=300, cost_quantiles=50),
            results_dir="./Results", figures_dir="./figures"),
        'quadratic': experiment_config(7, 'Synthetic nonlinear', model='quadratic',
            seeds=[42, 43, 44, 45, 46], alpha=0.4, B=0.75, lambda_=30.0,
            tau=0.01, eta=0.2, T=500,
            cohort_size=256, fit_cap=5000, synthetic_n=6000,
            audit_fractions=[0.05, 0.25, 0.5, 0.75, 0.95],
            audit_limit=32,
            baselines=dict(target_points=300, cost_quantiles=50),
            results_dir="./Results", figures_dir="./figures"),
    },

    'Synthetic curved': {
        'affine': experiment_config(7, 'Synthetic curved', model='affine',
            seeds=[42, 43, 44, 45, 46], alpha=0.4, B=0.75, lambda_=30.0,
            tau=0.01, eta=0.2, T=500,
            cohort_size=256, fit_cap=5000, synthetic_n=6000,
            audit_fractions=[0.05, 0.25, 0.5, 0.75, 0.95],
            audit_limit=32,
            baselines=dict(target_points=300, cost_quantiles=50),
            results_dir="./Results", figures_dir="./figures"),
        'quadratic': experiment_config(7, 'Synthetic curved', model='quadratic',
            seeds=[42, 43, 44, 45, 46], alpha=0.4, B=0.75, lambda_=30.0,
            tau=0.01, eta=0.2, T=500,
            cohort_size=256, fit_cap=5000, synthetic_n=6000,
            audit_fractions=[0.05, 0.25, 0.5, 0.75, 0.95],
            audit_limit=32,
            baselines=dict(target_points=300, cost_quantiles=50),
            results_dir="./Results", figures_dir="./figures"),
    },

}

# Selected pairs run sequentially; narrow the dataset list to run one dataset at a time.
experiment_7_selected_datasets = ['Synthetic curved', 'German Credit', 'Synthetic nonlinear', 'Give Me Some Credit', 'ACSIncome', 'Adult']
experiment_7_selected_models = ["affine", "quadratic"]

# Example local edits:
# experiment_7_selected_datasets = ["German Credit"]
# experiment_7_selected_models = ["affine"]
# experiment_7_configs["German Credit"]["quadratic"]["retry_failed"] = True


In [ ]:
experiment_7_mode = "run"  # "run", "preflight", "load", or "skip"
experiment_7_selected_configs = select_experiment_configs(
    experiment_7_configs,
    experiment_7_selected_datasets,
    experiment_7_selected_models,
)
experiment_7_run_configs = experiment_7_selected_configs
# A cheap plan: no data loading, fitting, or evaluation is triggered by this table.
display(pd.DataFrame([
    dict(dataset=config["dataset"], model=config["model"],
         lambda_values=(config["lambda_values"] if config["experiment"] == 2 else
                        [config["lambda_"]] if config["experiment"] != 7 else None),
         cases=len(experiment_cases(config)),
         policy_fit_requests=(0 if config["experiment"] == 7 else
             len(experiment_cases(config)) * sum(learned_method(m) is not None for m in config["methods"])))
    for config in experiment_7_run_configs
]))
experiment_7_reports = execute_experiment(
    experiment_7_run_configs,
    mode=experiment_7_mode,
)


## Experiment 8: Optional partial-adoption stress test

Train the personalized policy under full adoption separately for `lambda_values=[3, 30]` at `B=0.75`, `alpha=0.4`, and `tau=0.01`. Evaluate adoption probabilities `[0, 0.25, 0.5, 0.75, 1]` using 200 paired draws per seed. Recourse is not retrained for each adoption probability. Average draws within each seed before computing uncertainty across seeds. This stress test compares cost-sensitive and validity-sensitive recommendations; identical fitted policies from other experiments can be reused.

All six datasets have independent affine/quadratic configurations below. Choose datasets and models with the two selection lists.


In [ ]:
# Independent configurations: edit any dataset/model pair without affecting the others.
# lambda_values is the executed lambda grid (experiment 7 does not sweep lambda).
# lambda_ is a scalar base value; execution below expands the listed lambda values.
experiment_8_configs = {
    'German Credit': {
        'affine': experiment_config(8, 'German Credit', model='affine',
            seeds=[42, 43, 44, 45, 46], alpha=0.4, B=0.75, lambda_=30.0,
            tau=0.01, eta=0.2, T=500,
            cohort_size=256, fit_cap=5000, synthetic_n=6000,
            lambda_values=[3.0, 30.0],
            adoption_probabilities=[0.0, 0.25, 0.5, 0.75, 1.0],
            adoption_draws=200,
            baselines=dict(target_points=300, cost_quantiles=50),
            results_dir="./Results", figures_dir="./figures"),
        'quadratic': experiment_config(8, 'German Credit', model='quadratic',
            seeds=[42, 43, 44, 45, 46], alpha=0.4, B=0.75, lambda_=30.0,
            tau=0.01, eta=0.2, T=500,
            cohort_size=256, fit_cap=5000, synthetic_n=6000,
            lambda_values=[3.0, 30.0],
            adoption_probabilities=[0.0, 0.25, 0.5, 0.75, 1.0],
            adoption_draws=200,
            baselines=dict(target_points=300, cost_quantiles=50),
            results_dir="./Results", figures_dir="./figures"),
    },

    'Adult': {
        'affine': experiment_config(8, 'Adult', model='affine',
            seeds=[42, 43, 44, 45, 46], alpha=0.4, B=0.75, lambda_=30.0,
            tau=0.01, eta=0.2, T=500,
            cohort_size=256, fit_cap=5000, synthetic_n=6000,
            lambda_values=[3.0, 30.0],
            adoption_probabilities=[0.0, 0.25, 0.5, 0.75, 1.0],
            adoption_draws=200,
            baselines=dict(target_points=300, cost_quantiles=50),
            results_dir="./Results", figures_dir="./figures"),
        'quadratic': experiment_config(8, 'Adult', model='quadratic',
            seeds=[42, 43, 44, 45, 46], alpha=0.4, B=0.75, lambda_=30.0,
            tau=0.01, eta=0.2, T=500,
            cohort_size=256, fit_cap=5000, synthetic_n=6000,
            lambda_values=[3.0, 30.0],
            adoption_probabilities=[0.0, 0.25, 0.5, 0.75, 1.0],
            adoption_draws=200,
            baselines=dict(target_points=300, cost_quantiles=50),
            results_dir="./Results", figures_dir="./figures"),
    },

    'ACSIncome': {
        'affine': experiment_config(8, 'ACSIncome', model='affine',
            seeds=[42, 43, 44, 45, 46], alpha=0.4, B=0.75, lambda_=30.0,
            tau=0.01, eta=0.2, T=500,
            cohort_size=256, fit_cap=5000, synthetic_n=6000,
            lambda_values=[3.0, 30.0],
            adoption_probabilities=[0.0, 0.25, 0.5, 0.75, 1.0],
            adoption_draws=200,
            baselines=dict(target_points=300, cost_quantiles=50),
            results_dir="./Results", figures_dir="./figures"),
        'quadratic': experiment_config(8, 'ACSIncome', model='quadratic',
            seeds=[42, 43, 44, 45, 46], alpha=0.4, B=0.75, lambda_=30.0,
            tau=0.01, eta=0.2, T=500,
            cohort_size=256, fit_cap=5000, synthetic_n=6000,
            lambda_values=[3.0, 30.0],
            adoption_probabilities=[0.0, 0.25, 0.5, 0.75, 1.0],
            adoption_draws=200,
            baselines=dict(target_points=300, cost_quantiles=50),
            results_dir="./Results", figures_dir="./figures"),
    },

    'Give Me Some Credit': {
        'affine': experiment_config(8, 'Give Me Some Credit', model='affine',
            seeds=[42, 43, 44, 45, 46], alpha=0.4, B=0.75, lambda_=30.0,
            tau=0.01, eta=0.2, T=500,
            cohort_size=256, fit_cap=5000, synthetic_n=6000,
            lambda_values=[3.0, 30.0],
            adoption_probabilities=[0.0, 0.25, 0.5, 0.75, 1.0],
            adoption_draws=200,
            baselines=dict(target_points=300, cost_quantiles=50),
            gmsc_path='GiveMeSomeCredit.csv',
            results_dir="./Results", figures_dir="./figures"),
        'quadratic': experiment_config(8, 'Give Me Some Credit', model='quadratic',
            seeds=[42, 43, 44, 45, 46], alpha=0.4, B=0.75, lambda_=30.0,
            tau=0.01, eta=0.2, T=500,
            cohort_size=256, fit_cap=5000, synthetic_n=6000,
            lambda_values=[3.0, 30.0],
            adoption_probabilities=[0.0, 0.25, 0.5, 0.75, 1.0],
            adoption_draws=200,
            baselines=dict(target_points=300, cost_quantiles=50),
            gmsc_path='GiveMeSomeCredit.csv',
            results_dir="./Results", figures_dir="./figures"),
    },

    'Synthetic nonlinear': {
        'affine': experiment_config(8, 'Synthetic nonlinear', model='affine',
            seeds=[42, 43, 44, 45, 46], alpha=0.4, B=0.75, lambda_=30.0,
            tau=0.01, eta=0.2, T=500,
            cohort_size=256, fit_cap=5000, synthetic_n=6000,
            lambda_values=[3.0, 30.0],
            adoption_probabilities=[0.0, 0.25, 0.5, 0.75, 1.0],
            adoption_draws=200,
            baselines=dict(target_points=300, cost_quantiles=50),
            results_dir="./Results", figures_dir="./figures"),
        'quadratic': experiment_config(8, 'Synthetic nonlinear', model='quadratic',
            seeds=[42, 43, 44, 45, 46], alpha=0.4, B=0.75, lambda_=30.0,
            tau=0.01, eta=0.2, T=500,
            cohort_size=256, fit_cap=5000, synthetic_n=6000,
            lambda_values=[3.0, 30.0],
            adoption_probabilities=[0.0, 0.25, 0.5, 0.75, 1.0],
            adoption_draws=200,
            baselines=dict(target_points=300, cost_quantiles=50),
            results_dir="./Results", figures_dir="./figures"),
    },

    'Synthetic curved': {
        'affine': experiment_config(8, 'Synthetic curved', model='affine',
            seeds=[42, 43, 44, 45, 46], alpha=0.4, B=0.75, lambda_=30.0,
            tau=0.01, eta=0.2, T=500,
            cohort_size=256, fit_cap=5000, synthetic_n=6000,
            lambda_values=[3.0, 30.0],
            adoption_probabilities=[0.0, 0.25, 0.5, 0.75, 1.0],
            adoption_draws=200,
            baselines=dict(target_points=300, cost_quantiles=50),
            results_dir="./Results", figures_dir="./figures"),
        'quadratic': experiment_config(8, 'Synthetic curved', model='quadratic',
            seeds=[42, 43, 44, 45, 46], alpha=0.4, B=0.75, lambda_=30.0,
            tau=0.01, eta=0.2, T=500,
            cohort_size=256, fit_cap=5000, synthetic_n=6000,
            lambda_values=[3.0, 30.0],
            adoption_probabilities=[0.0, 0.25, 0.5, 0.75, 1.0],
            adoption_draws=200,
            baselines=dict(target_points=300, cost_quantiles=50),
            results_dir="./Results", figures_dir="./figures"),
    },

}

# Selected pairs run sequentially; narrow the dataset list to run one dataset at a time.
experiment_8_selected_datasets = ['Synthetic curved', 'German Credit', 'Synthetic nonlinear', 'Give Me Some Credit', 'ACSIncome', 'Adult']
experiment_8_selected_models = ["affine", "quadratic"]

# Example local edits:
# experiment_8_selected_datasets = ["German Credit"]
# experiment_8_selected_models = ["affine"]
# experiment_8_configs["German Credit"]["affine"]["lambda_values"] = [3.0, 30.0]
# experiment_8_configs["German Credit"]["quadratic"]["retry_failed"] = True


In [ ]:
experiment_8_mode = "run"  # "run", "preflight", "load", or "skip"
experiment_8_selected_configs = select_experiment_configs(
    experiment_8_configs,
    experiment_8_selected_datasets,
    experiment_8_selected_models,
)
# Each lambda is a separate saved run and plot; matching fitted policies are cached.
experiment_8_run_configs = [
    dict(copy.deepcopy(config), lambda_=float(lambda_))
    for config in experiment_8_selected_configs
    for lambda_ in config["lambda_values"]
]
# A cheap plan: no data loading, fitting, or evaluation is triggered by this table.
display(pd.DataFrame([
    dict(dataset=config["dataset"], model=config["model"],
         lambda_values=(config["lambda_values"] if config["experiment"] == 2 else
                        [config["lambda_"]] if config["experiment"] != 7 else None),
         cases=len(experiment_cases(config)),
         policy_fit_requests=(0 if config["experiment"] == 7 else
             len(experiment_cases(config)) * sum(learned_method(m) is not None for m in config["methods"])))
    for config in experiment_8_run_configs
]))
experiment_8_reports = execute_experiment(
    experiment_8_run_configs,
    mode=experiment_8_mode,
)


## Collect and export saved runs

The following functions read saved files without running experiments. `collect_saved_results` combines raw results across datasets, models, and configurations; keep `config_id` when grouping incompatible settings. `export_saved_runs` writes named per-configuration tables and regenerates all available plots in the shared `figures` folder, including for runs saved by older notebook versions. Dataset and model appear in plot titles; experiment numbers appear only in filenames. Experiments with no plot (such as the solver audit) still export their tables.

In [ ]:
# Collect all saved dataset/model/parameter settings without running experiments.
saved_results = collect_saved_results("./Results")
display(saved_results)

# Export/replot existing runs (including older notebook versions) without retraining.
# All plots go to this single folder; their titles contain only dataset and model.
exported = export_saved_runs("./Results", figures_dir="./figures")
display(exported)

# Optional filters, useful when exporting only one experiment or dataset/model:
exported = export_saved_runs("./Results", figures_dir="./figures", experiment=3)
exported = export_saved_runs("./Results", figures_dir="./figures",
                              dataset="German Credit", model="quadratic")